In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# %cd /kaggle/working

# !git clone https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
# %cd /kaggle/working/WeedsGalore_Ipr28

# !ls -lah
%cd /kaggle/working
!rm -rf WeedsGalore_Ipr28
!git clone -q https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
%cd /kaggle/working/WeedsGalore_Ipr28
!git log --oneline -1
REPO = '/kaggle/working/WeedsGalore_Ipr28'

/kaggle/working
/kaggle/working/WeedsGalore_Ipr28
fa64bdb (HEAD -> main, origin/main, origin/HEAD) Ignore weedmap_tiles_full


In [2]:
!pip install -q "transformers==4.45.2" torchmetrics absl-py opencv-python-headless pytest scikit-learn thop
import transformers, torchmetrics, sklearn
print('transformers', transformers.__version__, '| torchmetrics', torchmetrics.__version__, '| sklearn', sklearn.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 72.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.45.2 | torchmetrics 1.9.0 | sklearn 1.6.1


In [ ]:
# !pip install -q transformers==4.45.2

In [ ]:
# !pip install -q datasets huggingface_hub

In [ ]:
# from huggingface_hub import snapshot_download

# dataset_path = snapshot_download(
#     repo_id="Voxel51/weeds-galore",
#     repo_type="dataset",
#     local_dir="/kaggle/working/weeds-galore"
# )

# print(dataset_path)

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q absl-py==1.3.0 torchmetrics==0.11.4 tensorboard==2.11.0

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q --upgrade "tensorboard>=2.20,<2.21"

In [ ]:
# !cp -r /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset /kaggle/working/weedsgalore-dataset

In [ ]:
# !rm -rf /kaggle/working/weedsgalore-dataset

In [3]:
%cd /kaggle/working

!wget -O weedsgalore-dataset.zip \
"https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip"

/kaggle/working
--2026-11-05 12:31:37--  https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip
Resolving doidata.gfz.de (doidata.gfz.de)... 139.17.228.163
Connecting to doidata.gfz.de (doidata.gfz.de)|139.17.228.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 336989804 (321M) [application/zip]
Saving to: ‘weedsgalore-dataset.zip’

weedsgalore-dataset 100%[===================>] 321.38M  31.3MB/s    in 11s     

2026-11-05 12:31:49 (28.8 MB/s) - ‘weedsgalore-dataset.zip’ saved [336989804/336989804]



In [4]:
!unzip -q /kaggle/working/weedsgalore-dataset.zip \
    -d /kaggle/working/

In [5]:
import glob, os
# 1) Ưu tiên dataset đã Add Input (thư mục chứa các ngày 2023-05-25, ...)
hits = sorted(glob.glob('/kaggle/input/**/2023-05-25/semantics', recursive=True))
if hits:
    DS = os.path.dirname(os.path.dirname(hits[0]))
elif os.path.isdir('/kaggle/working/weedsgalore-dataset/2023-05-25'):
    DS = '/kaggle/working/weedsgalore-dataset'
else:
    # 2) Nếu chưa có: tải theo cách bạn đã dùng trước đây vào /kaggle/working/weedsgalore-dataset
    #    (trang dữ liệu: https://doidata.gfz.de/weedsgalore_e_celikkan_2024/)
    raise SystemExit('Chưa tìm thấy dataset: hãy Add Input hoặc tải về /kaggle/working/weedsgalore-dataset')
print('DS =', DS)
for d in sorted(os.listdir(DS)):
    p = os.path.join(DS, d)
    if os.path.isdir(p):
        print(d, sorted(os.listdir(p)))

DS = /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset
2023-05-25 ['images', 'instances', 'semantics']
2023-05-30 ['images', 'instances', 'semantics']
2023-06-06 ['images', 'instances', 'semantics']
2023-06-15 ['images', 'instances', 'semantics']
splits ['test.txt', 'train.txt', 'val.txt']


In [6]:
import inspect, os, sys
REPO = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(REPO)
from transformers.models.segformer.modeling_segformer import (
    SegformerEncoder, SegformerEfficientSelfAttention, SegformerOverlapPatchEmbeddings)
src_attn = inspect.getsource(SegformerEfficientSelfAttention.forward)
src_enc = inspect.getsource(SegformerEncoder.forward)
checks = {
    'Q computed before SR (query on un-reduced hidden_states)':
        src_attn.find('self.query(hidden_states)') < src_attn.find('self.sr('),
    'SR conv called on 4D tensor then layer_norm': 'self.sr(hidden_states)' in src_attn and 'self.layer_norm(hidden_states)' in src_attn,
    'encoder loop: embedding -> blocks -> norm -> reshape': all(s in src_enc for s in ['embedding_layer(hidden_states)', 'blk(hidden_states, height, width', 'norm_layer(hidden_states)']),
}
for k, v in checks.items():
    print('OK ' if v else 'FAIL', k)
os.makedirs('results/env', exist_ok=True)
with open('results/env/hf_source_check.txt', 'w') as f:
    f.write(src_attn + '\n\n' + src_enc)

import sys
if f'{REPO}/src' not in sys.path:
    sys.path.insert(0, f'{REPO}/src')
assert os.path.exists(f'{REPO}/src/nets/ccmc_sr.py'), 'Repo chưa có code mới: commit + push rồi chạy lại S2'
from nets.ccmc_sr import CCMCSegFormer
m = CCMCSegFormer(prior_mode='learned_weed')
for name, mod in m.named_modules():
    if name.endswith('.sr'):
        print(name, type(mod).__name__, tuple(mod.weight.shape), 'stride', mod.stride)
assert all(checks.values()), 'API HF khác giả định -> dừng lại, gửi file results/env/hf_source_check.txt để review'

OK  Q computed before SR (query on un-reduced hidden_states)
OK  SR conv called on 4D tensor then layer_norm
OK  encoder loop: embedding -> blocks -> norm -> reshape


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

encoder.encoder.block.0.0.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.0.1.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.1.0.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.1.1.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.2.0.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)
encoder.encoder.block.2.1.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)


In [7]:
%cd /kaggle/working/WeedsGalore_Ipr28
!mkdir -p results/env
!python -m pytest -q tests/test_ccmc_sr.py 2>&1 | tee results/env/unit_tests.log | tail -n 15

/kaggle/working
......................................................                   [100%]
54 passed in 34.61s


In [8]:
!pip -q install gdown

In [9]:
import os
import gdown

FILE_ID = '1tGelF2tv9u0hfkibTsIYD_NJRB2XGKjR'

DOWNLOAD_DIR = '/kaggle/working/weedmap-download'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

OUTPUT = os.path.join(DOWNLOAD_DIR, 'Tiles.zip')

if not os.path.isfile(OUTPUT) or os.path.getsize(OUTPUT) < 100_000_000:
    if os.path.exists(OUTPUT):
        os.remove(OUTPUT)

    result = gdown.download(
        id=FILE_ID,
        output=OUTPUT,
        quiet=False,
        fuzzy=True
    )

    assert result is not None, 'Google Drive download failed'

assert os.path.isfile(OUTPUT), 'Tiles.zip not found'
assert os.path.getsize(OUTPUT) > 100_000_000, 'Downloaded file is too small'

print('Download successful')
print('File size (GB):', round(os.path.getsize(OUTPUT) / 1024**3, 3))

Downloading...
From (original): https://drive.google.com/uc?id=1tGelF2tv9u0hfkibTsIYD_NJRB2XGKjR
From (redirected): https://drive.google.com/uc?id=1tGelF2tv9u0hfkibTsIYD_NJRB2XGKjR&confirm=t&uuid=72ec14ed-0141-4280-88d4-ad257b9ff715
To: /kaggle/working/weedmap-download/Tiles.zip
100%|██████████| 2.29G/2.29G [00:20<00:00, 114MB/s] 

Download successful
File size (GB): 2.137


In [10]:
import os
import zipfile

ARCHIVE = '/kaggle/working/weedmap-download/Tiles.zip'
WM_ROOT = '/kaggle/working/weedmap-2018'

os.makedirs(WM_ROOT, exist_ok=True)

with zipfile.ZipFile(ARCHIVE, 'r') as z:
    print('Files in archive:', len(z.namelist()))

    bad = z.testzip()
    assert bad is None, f'Corrupted ZIP file: {bad}'

    print('ZIP integrity: PASS')

    z.extractall(WM_ROOT)

print('Extraction completed:', WM_ROOT)


Files in archive: 2679
ZIP integrity: PASS
Extraction completed: /kaggle/working/weedmap-2018


In [11]:
import os
import zipfile

ZIP_PATH = '/kaggle/working/weedmap-2018/Tiles/RedEdge/003.zip'
OUT_DIR = '/kaggle/working/weedmap-2018/Tiles/RedEdge/003'

assert os.path.isfile(ZIP_PATH), f'ZIP not found: {ZIP_PATH}'

os.makedirs(OUT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(OUT_DIR)

print('Extraction completed!')
print('Output:', OUT_DIR)
print('Top-level contents:', os.listdir(OUT_DIR))

Extraction completed!
Output: /kaggle/working/weedmap-2018/Tiles/RedEdge/003
Top-level contents: ['__MACOSX', '003']


In [12]:
#Cell chỉ tìm checkpoint và run-config, không đọc dữ liệu test, không chạy inference
from pathlib import Path
import subprocess
import hashlib
import json

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
assert PROJECT.is_dir(), f'Missing project: {PROJECT}'

head = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'],
    text=True
).strip()
print('Kaggle HEAD:', head)

models = ('L0', 'OccCondg', 'MCSDg')
search_roots = [
    PROJECT / 'runs',
    Path('/kaggle/input'),
]

found = {}
for model in models:
    for seed in range(5):
        run_name = f'{model}_s{seed}'
        found[run_name] = {}

        for filename in ('best_val.pth', 'run_config.json'):
            matches = []
            for root in search_roots:
                if root.is_dir():
                    matches.extend(
                        p for p in root.rglob(filename)
                        if p.parent.name == run_name
                    )
            found[run_name][filename] = [
                str(p) for p in sorted(set(matches))
            ]
            print(run_name, filename, ':',
                  len(found[run_name][filename]), 'matches')

out = PROJECT / 'results/exp08'
out.mkdir(parents=True, exist_ok=True)
report = out / 'checkpoint_search_kaggle.json'
report.write_text(
    json.dumps({'commit': head, 'found': found}, indent=2),
    encoding='utf-8'
)

missing = [
    name for name, files in found.items()
    if not files['best_val.pth']
]
print('\nMissing checkpoint runs:', missing)
print('Saved:', report)
print('NO TEST INFERENCE PERFORMED')


Kaggle HEAD: fa64bdbe992d990a84133a487e9780c7c784ac69
L0_s0 best_val.pth : 1 matches
L0_s0 run_config.json : 0 matches
L0_s1 best_val.pth : 1 matches
L0_s1 run_config.json : 0 matches
L0_s2 best_val.pth : 1 matches
L0_s2 run_config.json : 0 matches
L0_s3 best_val.pth : 1 matches
L0_s3 run_config.json : 0 matches
L0_s4 best_val.pth : 1 matches
L0_s4 run_config.json : 0 matches
OccCondg_s0 best_val.pth : 1 matches
OccCondg_s0 run_config.json : 0 matches
OccCondg_s1 best_val.pth : 1 matches
OccCondg_s1 run_config.json : 0 matches
OccCondg_s2 best_val.pth : 1 matches
OccCondg_s2 run_config.json : 0 matches
OccCondg_s3 best_val.pth : 1 matches
OccCondg_s3 run_config.json : 0 matches
OccCondg_s4 best_val.pth : 1 matches
OccCondg_s4 run_config.json : 0 matches
MCSDg_s0 best_val.pth : 1 matches
MCSDg_s0 run_config.json : 0 matches
MCSDg_s1 best_val.pth : 1 matches
MCSDg_s1 run_config.json : 0 matches
MCSDg_s2 best_val.pth : 1 matches
MCSDg_s2 run_config.json : 0 matches
MCSDg_s3 best_val.pth :

In [13]:
#Cell dưới đây tìm run_config.json trong repository và Kaggle Input, đối chiếu seed, method và split hash, đồng thời tính SHA-256 của 15 checkpoint.
from pathlib import Path
import json
import hashlib
import subprocess

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
assert PROJECT.is_dir()
OUT = PROJECT / 'results/exp08'
OUT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

head = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'],
    text=True
).strip()

expected_prior = 'fe3daf1a91203b20d73aac960d08f2dbf96be872'
ancestry = subprocess.run(
    ['git', '-C', str(PROJECT), 'merge-base',
     '--is-ancestor', expected_prior, head]
)
print('Current HEAD:', head)
print('Prior commit is ancestor:', ancestry.returncode == 0)

SEARCH_ROOTS = [
    PROJECT / 'runs',
    PROJECT / 'doc',
    Path('/kaggle/input'),
]

expected_split_hash = '675667edf3f35f16'
report = {'commit': head, 'runs': {}}

for method in ('L0', 'OccCondg', 'MCSDg'):
    for seed in range(5):
        run = f'{method}_s{seed}'
        matches = {'best_val.pth': [], 'run_config.json': []}

        for root in SEARCH_ROOTS:
            if not root.exists():
                continue
            for filename in matches:
                for p in root.rglob(filename):
                    if p.parent.name == run:
                        matches[filename].append(p)

        row = {}
        for filename, paths in matches.items():
            unique = sorted(set(paths), key=str)
            row[filename] = [
                {'path': str(p), 'sha256': sha256(p),
                 'bytes': p.stat().st_size}
                for p in unique
            ]

        row['config_validation'] = []
        for item in row['run_config.json']:
            path = Path(item['path'])
            try:
                cfg = json.loads(path.read_text(encoding='utf-8'))
                flags = cfg.get('flags', {})
                row['config_validation'].append({
                    'path': str(path),
                    'split_hash_matches':
                        cfg.get('split_hash') == expected_split_hash,
                    'seed_matches':
                        str(flags.get('seed')) == str(seed),
                    'head': flags.get('head'),
                    'ccmc': flags.get('ccmc'),
                    'spectral_guide': flags.get('spectral_guide')
                })
            except Exception as exc:
                row['config_validation'].append({
                    'path': str(path), 'error': str(exc)
                })

        report['runs'][run] = row
        print(run,
              'ckpts:', len(row['best_val.pth']),
              'configs:', len(row['run_config.json']))

report_path = OUT / 'checkpoint_config_provenance_audit.json'
report_path.write_text(
    json.dumps(report, indent=2), encoding='utf-8'
)

print('\nSaved:', report_path)
print('AUDIT ONLY — NO INFERENCE, NO TEST RELEASE')


Current HEAD: fa64bdbe992d990a84133a487e9780c7c784ac69
Prior commit is ancestor: True
L0_s0 ckpts: 1 configs: 1
L0_s1 ckpts: 1 configs: 1
L0_s2 ckpts: 1 configs: 1
L0_s3 ckpts: 1 configs: 1
L0_s4 ckpts: 1 configs: 1
OccCondg_s0 ckpts: 1 configs: 1
OccCondg_s1 ckpts: 1 configs: 1
OccCondg_s2 ckpts: 1 configs: 1
OccCondg_s3 ckpts: 1 configs: 1
OccCondg_s4 ckpts: 1 configs: 1
MCSDg_s0 ckpts: 1 configs: 1
MCSDg_s1 ckpts: 1 configs: 1
MCSDg_s2 ckpts: 1 configs: 1
MCSDg_s3 ckpts: 1 configs: 1
MCSDg_s4 ckpts: 1 configs: 1

Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp08/checkpoint_config_provenance_audit.json
AUDIT ONLY — NO INFERENCE, NO TEST RELEASE


In [14]:
# Chạy cell K8.0n trên Kaggle
# Cell này dùng chính file audit K8.0m, không tự suy đoán đường dẫn và không ghi đè checkpoint.

from pathlib import Path
import hashlib, json, shutil, os, subprocess

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
os.chdir(PROJECT)

AUDIT = PROJECT / 'results/exp08/checkpoint_config_provenance_audit.json'
assert AUDIT.is_file(), 'Run K8.0m first'
audit = json.loads(AUDIT.read_text(encoding='utf-8'))

HEAD = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()
assert audit['commit'] == HEAD, 'Commit changed after audit'

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

expected_runs = {
    f'{method}_s{seed}'
    for method in ('L0', 'OccCondg', 'MCSDg')
    for seed in range(5)
}
assert set(audit['runs']) == expected_runs

rows = {}
for run in sorted(expected_runs):
    entry = audit['runs'][run]
    ckpts = entry['best_val.pth']
    configs = entry['run_config.json']
    assert len(ckpts) == len(configs) == 1, run

    src_ckpt = Path(ckpts[0]['path'])
    src_cfg = Path(configs[0]['path'])
    assert src_ckpt.is_file() and src_cfg.is_file(), run
    assert sha256(src_ckpt) == ckpts[0]['sha256']
    assert sha256(src_cfg) == configs[0]['sha256']

    cfg = json.loads(src_cfg.read_text(encoding='utf-8'))
    flags = cfg['flags']
    method, seed_str = run.rsplit('_s', 1)
    seed = int(seed_str)

    assert cfg['split_hash'] == '675667edf3f35f16'
    assert int(flags['seed']) == seed
    assert bool(flags['ccmc']) == (method == 'L0')

    if method != 'L0':
        expected_head = {
            'OccCondg': 'occ_conditioned',
            'MCSDg': 'mcsd'
        }[method]
        assert flags['head'] == expected_head
        assert bool(flags['spectral_guide']) is True
    elif 'head' in flags:
        assert flags['head'] == 'bilinear'
        assert bool(flags.get('spectral_guide', False)) is False

    dest = PROJECT / 'runs' / run
    dest.mkdir(parents=True, exist_ok=True)

    dest_ckpt = dest / 'best_val.pth'
    dest_cfg = dest / 'run_config.json'

    # Link checkpoint goc; khong sao chep hay ghi de weights
    if dest_ckpt.exists() or dest_ckpt.is_symlink():
        assert dest_ckpt.is_file()
        assert sha256(dest_ckpt) == sha256(src_ckpt)
    else:
        dest_ckpt.symlink_to(src_ckpt)

    # Sao chep nguyen van run_config goc
    if dest_cfg.exists():
        assert sha256(dest_cfg) == sha256(src_cfg)
    else:
        shutil.copy2(src_cfg, dest_cfg)

    rows[run] = {
        'checkpoint_source': str(src_ckpt),
        'checkpoint_sha256': sha256(src_ckpt),
        'config_source': str(src_cfg),
        'config_sha256': sha256(src_cfg),
        'legacy_L0_config': method == 'L0' and 'head' not in flags,
        'verified': True,
    }
    print(run, 'OK')

result = {
    'status': 'CHECKPOINT_PROVENANCE_PASS',
    'test_authorization': False,
    'commit': HEAD,
    'checkpoint_count': len(rows),
    'runs': rows,
    'pending': [
        'WeedMap mask validity',
        'WeedMap radiometry',
        'checkpoint synthetic forward integration',
        'reviewed preflight and evaluator acceptance',
        'immutable test manifest'
    ]
}

out = PROJECT / 'results/exp08/checkpoint_provenance_verified.json'
out.write_text(json.dumps(result, indent=2), encoding='utf-8')

print('\n15/15 checkpoint/config pairs staged and verified')
print('Saved:', out)
print('NO INFERENCE — K8.2/K8.3 STILL NO-GO')


L0_s0 OK
L0_s1 OK
L0_s2 OK
L0_s3 OK
L0_s4 OK
MCSDg_s0 OK
MCSDg_s1 OK
MCSDg_s2 OK
MCSDg_s3 OK
MCSDg_s4 OK
OccCondg_s0 OK
OccCondg_s1 OK
OccCondg_s2 OK
OccCondg_s3 OK
OccCondg_s4 OK

15/15 checkpoint/config pairs staged and verified
Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp08/checkpoint_provenance_verified.json
NO INFERENCE — K8.2/K8.3 STILL NO-GO


In [15]:
#Cell K8.0o — chạy trên Kaggle
#Cell này kiểm tra cấu trúc checkpoint, tensor weights và metadata. Không chạy inference trên WeedMap hoặc WeedsGalore, không sửa checkpoint.

import os, json, hashlib, subprocess
from pathlib import Path
import torch

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
os.chdir(PROJECT)

OUT = PROJECT / 'results/exp08'
OUT.mkdir(parents=True, exist_ok=True)

provenance_path = OUT / 'checkpoint_provenance_verified.json'
assert provenance_path.is_file(), 'K8.0n report missing'

provenance = json.loads(
    provenance_path.read_text(encoding='utf-8')
)
assert provenance['status'] == 'CHECKPOINT_PROVENANCE_PASS'
assert provenance['checkpoint_count'] == 15
assert provenance['test_authorization'] is False

head = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()
assert head == provenance['commit'], 'Git HEAD changed'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

report = {
    'status': 'PENDING',
    'commit': head,
    'test_authorization': False,
    'runs': {}
}

for run, item in sorted(provenance['runs'].items()):
    path = PROJECT / 'runs' / run / 'best_val.pth'
    assert path.is_file(), f'Missing: {path}'
    assert sha256(path) == item['checkpoint_sha256']

    checkpoint = torch.load(
        path, map_location='cpu', weights_only=True
    )

    if not isinstance(checkpoint, dict):
        raise TypeError(f'{run}: checkpoint is not a dict')

    candidates = [
        ('model_state_dict', checkpoint.get('model_state_dict')),
        ('state_dict', checkpoint.get('state_dict')),
        ('model', checkpoint.get('model')),
        ('direct', checkpoint)
    ]

    state_name = None
    state = None
    for name, candidate in candidates:
        if isinstance(candidate, dict) and candidate:
            if all(torch.is_tensor(v) for v in candidate.values()):
                state_name = name
                state = candidate
                break

    assert state is not None, (
        f'{run}: cannot identify tensor state_dict; '
        f'top-level keys={list(checkpoint)[:20]}'
    )

    n_params = sum(v.numel() for v in state.values())
    nonfinite = [
        key for key, value in state.items()
        if value.is_floating_point()
        and not torch.isfinite(value).all()
    ]
    assert not nonfinite, f'{run}: nonfinite tensors: {nonfinite[:5]}'

    report['runs'][run] = {
        'checkpoint_sha256': sha256(path),
        'state_dict_key': state_name,
        'state_tensor_count': len(state),
        'state_element_count': n_params,
        'first_tensor_shapes': {
            k: list(v.shape)
            for k, v in list(state.items())[:12]
        },
        'top_level_keys': list(checkpoint.keys())[:30],
        'nonfinite_tensor_count': 0,
        'status': 'PASS'
    }

    print(run, 'PASS',
          'tensors=', len(state),
          'elements=', n_params,
          'state=', state_name)

report['status'] = 'CHECKPOINT_STRUCTURE_PASS'
report['passed'] = len(report['runs'])
report['pending'] = [
    'strict state_dict loading into matching model architecture',
    'synthetic forward/output-size integration',
    'WeedMap mask provenance',
    'WeedMap radiometric compatibility',
    'reviewed preflight and final protocol freeze'
]

output = OUT / 'checkpoint_structure_audit.json'
output.write_text(
    json.dumps(report, indent=2), encoding='utf-8'
)

print('\nSaved:', output)
print('Passed:', report['passed'], '/15')
print('NO TEST INFERENCE PERFORMED')
print('K8.2/K8.3 STILL NO-GO')


L0_s0 PASS tensors= 365 elements= 3456289 state= direct
L0_s1 PASS tensors= 365 elements= 3456289 state= direct
L0_s2 PASS tensors= 365 elements= 3456289 state= direct
L0_s3 PASS tensors= 365 elements= 3456289 state= direct
L0_s4 PASS tensors= 365 elements= 3456289 state= direct
MCSDg_s0 PASS tensors= 375 elements= 3463317 state= direct
MCSDg_s1 PASS tensors= 375 elements= 3463317 state= direct
MCSDg_s2 PASS tensors= 375 elements= 3463317 state= direct
MCSDg_s3 PASS tensors= 375 elements= 3463317 state= direct
MCSDg_s4 PASS tensors= 375 elements= 3463317 state= direct
OccCondg_s0 PASS tensors= 375 elements= 3463317 state= direct
OccCondg_s1 PASS tensors= 375 elements= 3463317 state= direct
OccCondg_s2 PASS tensors= 375 elements= 3463317 state= direct
OccCondg_s3 PASS tensors= 375 elements= 3463317 state= direct
OccCondg_s4 PASS tensors= 375 elements= 3463317 state= direct

Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp08/checkpoint_structure_audit.json
Passed: 15 /15
NO TEST INFE

In [16]:
#Chạy K8.0p — kiểm toán THOP và chuẩn bị strict loading
#Chạy cell này trên Kaggle, ngay sau K8.0o. Cell không đọc ảnh test, không forward model và không thay đổi trọng số gốc.

from pathlib import Path
import os, json, hashlib, subprocess, re
import torch

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
os.chdir(PROJECT)
OUT = PROJECT / 'results/exp08'

source = json.loads(
    (OUT / 'checkpoint_provenance_verified.json')
    .read_text(encoding='utf-8')
)
structure = json.loads(
    (OUT / 'checkpoint_structure_audit.json')
    .read_text(encoding='utf-8')
)

HEAD = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()

assert HEAD == source['commit'] == structure['commit']
assert source['checkpoint_count'] == structure['passed'] == 15

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

thop_pattern = re.compile(
    r'(^|\.)(total_ops|total_params)$'
)

report = {
    'status': 'PENDING',
    'commit': HEAD,
    'scope': 'THOP buffer audit only; no model forward',
    'test_authorization': False,
    'runs': {}
}

for run, record in sorted(source['runs'].items()):
    path = Path(record['checkpoint_source'])
    assert path.is_file()
    assert sha256(path) == record['checkpoint_sha256']

    state = torch.load(
        path, map_location='cpu', weights_only=True
    )
    assert isinstance(state, dict)
    assert all(torch.is_tensor(v) for v in state.values())

    thop_keys = sorted(
        k for k in state if thop_pattern.search(k)
    )
    normal_keys = sorted(
        k for k in state if not thop_pattern.search(k)
    )

    assert normal_keys, f'{run}: no model tensors'
    assert all(
        state[k].numel() == 1 for k in thop_keys
    ), f'{run}: unexpected THOP-buffer shape'

    # Chỉ tạo state sạch trong RAM; không save checkpoint.
    cleaned = {
        k: v for k, v in state.items()
        if k in normal_keys
    }

    assert not any(
        thop_pattern.search(k) for k in cleaned
    )

    report['runs'][run] = {
        'checkpoint_sha256': sha256(path),
        'original_tensor_count': len(state),
        'thop_buffer_count': len(thop_keys),
        'model_tensor_count': len(cleaned),
        'model_element_count': sum(
            v.numel() for v in cleaned.values()
        ),
        'thop_keys': thop_keys,
        'status': 'THOP_KEYS_CLASSIFIED'
    }

    print(
        run,
        '| THOP buffers:', len(thop_keys),
        '| model tensors:', len(cleaned)
    )

report['status'] = 'THOP_BUFFER_AUDIT_PASS'
report['passed'] = len(report['runs'])
report['pending'] = [
    'strict load cleaned state into exact model',
    'synthetic model forward',
    'mask policy provenance',
    'radiometry provenance',
    'protocol freeze'
]

path = OUT / 'checkpoint_thop_buffer_audit.json'
path.write_text(
    json.dumps(report, indent=2),
    encoding='utf-8'
)

print('\nPASS:', report['passed'], '/15')
print('Saved:', path)
print('No checkpoint modified; no test inference')


L0_s0 | THOP buffers: 156 | model tensors: 209
L0_s1 | THOP buffers: 156 | model tensors: 209
L0_s2 | THOP buffers: 156 | model tensors: 209
L0_s3 | THOP buffers: 156 | model tensors: 209
L0_s4 | THOP buffers: 156 | model tensors: 209
MCSDg_s0 | THOP buffers: 156 | model tensors: 219
MCSDg_s1 | THOP buffers: 156 | model tensors: 219
MCSDg_s2 | THOP buffers: 156 | model tensors: 219
MCSDg_s3 | THOP buffers: 156 | model tensors: 219
MCSDg_s4 | THOP buffers: 156 | model tensors: 219
OccCondg_s0 | THOP buffers: 156 | model tensors: 219
OccCondg_s1 | THOP buffers: 156 | model tensors: 219
OccCondg_s2 | THOP buffers: 156 | model tensors: 219
OccCondg_s3 | THOP buffers: 156 | model tensors: 219
OccCondg_s4 | THOP buffers: 156 | model tensors: 219

PASS: 15 /15
Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp08/checkpoint_thop_buffer_audit.json
No checkpoint modified; no test inference


In [17]:
# K8.0q — chuẩn bị strict loading
# Trước khi kiểm thử forward, cần lấy chính xác cách repository khởi tạo mô hình. Tôi không muốn đoán constructor vì L0, OccCond-g và MCSD-g sử dụng các cờ kiến trúc khác nhau.
# Chạy cell sau trên Kaggle. Nó tìm các đoạn code xây dựng model và chữ ký constructor, không đọc dữ liệu test.

from pathlib import Path
import ast, json, os, subprocess

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
os.chdir(PROJECT)
OUT = PROJECT / 'results/exp08'
OUT.mkdir(parents=True, exist_ok=True)

files = [
    PROJECT / 'src/evaluate_ccmc.py',
    PROJECT / 'src/train.py',
    PROJECT / 'src/nets/mcsd.py',
    PROJECT / 'src/nets/segformer.py',
]

report = {
    'status': 'MODEL_CONSTRUCTION_SOURCE_AUDIT',
    'commit': subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], text=True
    ).strip(),
    'test_authorization': False,
    'files': {}
}

for path in files:
    assert path.is_file(), f'Missing source: {path}'
    lines = path.read_text(encoding='utf-8').splitlines()
    tree = ast.parse('\n'.join(lines), filename=str(path))

    selected = []

    for node in ast.walk(tree):
        if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
            name = node.name.lower()
            if any(x in name for x in (
                'model', 'segformer', 'mcsd',
                'build', 'create', 'load'
            )):
                selected.append({
                    'kind': type(node).__name__,
                    'name': node.name,
                    'line': node.lineno,
                    'source': '\n'.join(
                        lines[node.lineno - 1:
                              min(node.lineno + 18, len(lines))]
                    )
                })

        elif isinstance(node, ast.Call):
            expr = ast.get_source_segment(
                '\n'.join(lines), node.func
            ) or ''
            if any(x in expr for x in (
                'MCSDSegFormer', 'SegFormer',
                'load_state_dict'
            )):
                selected.append({
                    'kind': 'Call',
                    'name': expr,
                    'line': node.lineno,
                    'source': '\n'.join(
                        lines[max(0, node.lineno - 5):
                              min(node.end_lineno + 4, len(lines))]
                    )
                })

    report['files'][str(path.relative_to(PROJECT))] = selected

    print('\nFILE:', path.relative_to(PROJECT))
    for item in selected[:18]:
        print('\n', item['kind'], item['name'],
              'line', item['line'])
        print(item['source'][:1600])

output = OUT / 'model_construction_source_audit.json'
output.write_text(json.dumps(report, indent=2),
                  encoding='utf-8')

print('\nSaved:', output)
print('SOURCE AUDIT ONLY — NO MODEL INFERENCE')



FILE: src/evaluate_ccmc.py

FILE: src/train.py

 FunctionDef measure_model_stats line 183
def measure_model_stats(model, device, in_channels, input_size=(512, 512)):
    """Measure model parameters, FLOPs, and inference time"""
    # Count parameters
    total_params, trainable_params, non_trainable_params = count_parameters(model)
    print(f"\n{'='*60}")
    print(f"MODEL STATISTICS")
    print(f"{'='*60}")
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Non-trainable Parameters: {non_trainable_params:,}")
   
    # Measure FLOPs
    if THOP_AVAILABLE:
        try:
            dummy_input = torch.randn(1, in_channels, input_size[0], input_size[1]).to(device)
            flops, params = profile(model, inputs=(dummy_input,), verbose=False)
            print(f"FLOPs (forward pass): {flops:,.0f} (~{flops/1e9:.2f}G)")
            print(f"FLOPs per pixel: {flops / (input_size[0] * input_size[1]):,.0f}")
        except E

In [18]:
# Cell K8.0r — strict loading + synthetic forward trên Kaggle
# Chạy nguyên cell dưới đây. Nó kiểm thử 15 checkpoint, nhưng không đọc WeedMap, không đọc WeedsGalore, không ghi dự đoán.

import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import ast
import hashlib
import importlib
import inspect
import json
import re
import subprocess
import sys
import traceback
from pathlib import Path

import torch

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
assert PROJECT.is_dir()
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

OUT = PROJECT / "results/exp08"
provenance = json.loads(
    (OUT / "checkpoint_provenance_verified.json")
    .read_text(encoding="utf-8")
)
thop_audit = json.loads(
    (OUT / "checkpoint_thop_buffer_audit.json")
    .read_text(encoding="utf-8")
)

head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

assert head == provenance["commit"] == thop_audit["commit"]
assert provenance["checkpoint_count"] == 15
assert thop_audit["passed"] == 15

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_class(class_name):
    # Search source code, not training/evaluation entrypoints.
    matches = []
    for path in (PROJECT / "src/nets").rglob("*.py"):
        tree = ast.parse(
            path.read_text(encoding="utf-8"),
            filename=str(path)
        )
        if any(
            isinstance(n, ast.ClassDef) and n.name == class_name
            for n in ast.walk(tree)
        ):
            matches.append(path)

    assert len(matches) == 1, (
        class_name, "class definitions:", matches
    )

    relative = matches[0].relative_to(PROJECT).with_suffix("")
    module_name = ".".join(relative.parts)
    module = importlib.import_module(module_name)
    return getattr(module, class_name)

def require(flags, name):
    assert name in flags, f"Missing frozen config flag: {name}"
    return flags[name]

def as_bool(value):
    if isinstance(value, bool):
        return value
    if value in (0, 1):
        return bool(value)
    if isinstance(value, str):
        if value.lower() in ("true", "1"):
            return True
        if value.lower() in ("false", "0"):
            return False
    raise ValueError(f"Invalid boolean value: {value!r}")

def stages_as_ints(value):
    if isinstance(value, (list, tuple)):
        return [int(v) for v in value]
    if isinstance(value, str):
        return [
            int(v) for v in
            value.replace(",", " ").split()
        ]
    raise ValueError(f"Invalid CCMC stages: {value!r}")

def build_model(run, flags):
    method = run.rsplit("_s", 1)[0]
    common = dict(
        num_classes=int(require(flags, "num_classes")),
        in_channels=int(require(flags, "in_channels")),
        use_multiscale_fusion=as_bool(
            require(flags, "use_multiscale_fusion")
        )
    )
    assert common["num_classes"] == 3
    assert common["in_channels"] == 5
    assert common["use_multiscale_fusion"] is True

    if method == "L0":
        assert as_bool(require(flags, "ccmc")) is True
        cls = find_class("CCMCSegFormer")

        # Mirrors the CCMCSegFormer branch in src/train.py.
        mapping = {
            "use_low_level_refine": "use_low_level_refine",
            "prior_mode": "prior_mode",
            "injection": "injection",
            "prior_perturb": "prior_perturb",
            "stages": "ccmc_stages",
            "eps": "ccmc_eps",
            "lam_init": "lam_init",
            "learn_lambda": "learn_lambda",
        }
        kwargs = dict(common)
        defaults_used = []
        sig = inspect.signature(cls)

        for param, flag in mapping.items():
            if flag in flags:
                value = flags[flag]
                if param == "stages":
                    value = stages_as_ints(value)
                elif param in (
                    "use_low_level_refine", "learn_lambda"
                ):
                    value = as_bool(value)
                kwargs[param] = value
            else:
                p = sig.parameters.get(param)
                assert p is not None, (
                    f"Constructor missing parameter {param}"
                )
                assert p.default is not inspect.Parameter.empty, (
                    f"No config or default for {param}"
                )
                defaults_used.append(param)

        return cls(**kwargs), defaults_used

    assert as_bool(require(flags, "ccmc")) is False
    expected_head = {
        "OccCondg": "occ_conditioned",
        "MCSDg": "mcsd",
    }[method]
    assert require(flags, "head") == expected_head
    assert as_bool(require(flags, "spectral_guide")) is True

    cls = find_class("MCSDSegFormer")
    kwargs = {
        **common,
        "head": expected_head,
        "sinkhorn_iters": int(
            require(flags, "sinkhorn_iters")
        ),
        "tau": float(require(flags, "mcsd_tau")),
        "spectral_guide": True
    }
    return cls(**kwargs), []

thop_pattern = re.compile(
    r"(^|\.)(total_ops|total_params)$"
)

report = {
    "status": "IN_PROGRESS",
    "commit": head,
    "scope": "Strict checkpoint load and synthetic forward only",
    "test_authorization": False,
    "input_shape": [1, 5, 128, 128],
    "runs": {}
}

device = torch.device("cpu")

for run, item in sorted(provenance["runs"].items()):
    print("CHECK:", run, flush=True)
    row = {"status": "FAIL"}

    try:
        ckpt = Path(item["checkpoint_source"])
        config_path = Path(item["config_source"])

        assert sha256(ckpt) == item["checkpoint_sha256"]
        assert sha256(config_path) == item["config_sha256"]

        config = json.loads(
            config_path.read_text(encoding="utf-8")
        )
        assert config["split_hash"] == "675667edf3f35f16"

        model, defaults_used = build_model(
            run, config["flags"]
        )

        state = torch.load(
            ckpt, map_location="cpu", weights_only=True
        )
        clean_state = {
            k: v for k, v in state.items()
            if not thop_pattern.search(k)
        }

        assert len(clean_state) == (
            thop_audit["runs"][run]["model_tensor_count"]
        )

        # Do not use strict=False.
        model.load_state_dict(clean_state, strict=True)
        model.to(device).eval()

        x = torch.zeros(
            1, 5, 128, 128,
            dtype=torch.float32,
            device=device
        )

        with torch.inference_mode():
            y = model(x)

        if isinstance(y, (tuple, list)):
            y = y[0]

        if hasattr(y, "logits"):
            y = y.logits

        assert torch.is_tensor(y), type(y)
        assert tuple(y.shape) == (1, 3, 128, 128), (
            f"Unexpected output shape: {tuple(y.shape)}"
        )
        assert torch.isfinite(y).all().item()

        row = {
            "status": "PASS",
            "strict_load": True,
            "input_shape": [1, 5, 128, 128],
            "output_shape": list(y.shape),
            "output_finite": True,
            "checkpoint_sha256": sha256(ckpt),
            "config_sha256": sha256(config_path),
            "constructor_defaults_used": defaults_used
        }
        print(run, "PASS", flush=True)

        del model, state, clean_state, x, y

    except Exception:
        row["error"] = traceback.format_exc()
        print(run, "FAIL", flush=True)
        print(row["error"][-1500:], flush=True)

    report["runs"][run] = row

report["passed"] = sum(
    r["status"] == "PASS"
    for r in report["runs"].values()
)
report["failed"] = 15 - report["passed"]

fallback_runs = {
    run: r.get("constructor_defaults_used", [])
    for run, r in report["runs"].items()
    if r.get("constructor_defaults_used")
}
report["constructor_default_review_required"] = fallback_runs

report["status"] = (
    "SYNTHETIC_FORWARD_PASS"
    if report["passed"] == 15 and not fallback_runs
    else "REVIEW_REQUIRED"
)

output = OUT / "checkpoint_synthetic_forward_audit.json"
output.write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

print("\nPassed:", report["passed"], "/15")
print("Failed:", report["failed"])
print("Constructor defaults to review:", fallback_runs)
print("Saved:", output)
print("NO REAL-DATA INFERENCE; K8.2/K8.3 NOT AUTHORIZED")


CHECK: L0_s0
L0_s0 PASS
CHECK: L0_s1
L0_s1 PASS
CHECK: L0_s2
L0_s2 PASS
CHECK: L0_s3
L0_s3 PASS
CHECK: L0_s4
L0_s4 PASS
CHECK: MCSDg_s0
MCSDg_s0 PASS
CHECK: MCSDg_s1
MCSDg_s1 PASS
CHECK: MCSDg_s2
MCSDg_s2 PASS
CHECK: MCSDg_s3
MCSDg_s3 PASS
CHECK: MCSDg_s4
MCSDg_s4 PASS
CHECK: OccCondg_s0
OccCondg_s0 PASS
CHECK: OccCondg_s1
OccCondg_s1 PASS
CHECK: OccCondg_s2
OccCondg_s2 PASS
CHECK: OccCondg_s3
OccCondg_s3 PASS
CHECK: OccCondg_s4
OccCondg_s4 PASS

Passed: 15 /15
Failed: 0
Constructor defaults to review: {}
Saved: /kaggle/working/WeedsGalore_Ipr28/results/exp08/checkpoint_synthetic_forward_audit.json
NO REAL-DATA INFERENCE; K8.2/K8.3 NOT AUTHORIZED


In [15]:
#Cell K8.0 — WeedMap RedEdge-M 003 inventory (no inference)
import glob, hashlib, json, os, pathlib, sys
from collections import Counter
from PIL import Image

PROJECT = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(PROJECT)
EXP = 'exp08'
OUT = f'results/{EXP}'
os.makedirs(f'{OUT}/logs', exist_ok=True)

WM_CANDIDATES = ['/kaggle/working/weedmap-2018'] + glob.glob(
    '/kaggle/input/**/weedmap-2018', recursive=True)
WM_ROOT = next((path for path in WM_CANDIDATES if os.path.isdir(path)), None)
assert WM_ROOT, f'WeedMap root not found: {WM_CANDIDATES}'
WM_ROOT = pathlib.Path(WM_ROOT)
INDEX_ROOT = WM_ROOT / 'Tiles' / 'RedEdge' / '003'
INDEX_ARCHIVE = WM_ROOT / 'Tiles' / 'RedEdge' / '003.zip'
assert INDEX_ROOT.is_dir(), f'Extracted RedEdge index 003 not found: {INDEX_ROOT}'
assert INDEX_ARCHIVE.is_file(), f'RedEdge index 003 archive not found: {INDEX_ARCHIVE}'

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

all_files = [p for p in INDEX_ROOT.rglob('*') if p.is_file()]
print('Mounted root:', WM_ROOT)
print('Dataset index root:', INDEX_ROOT)
print('Index 003 archive sha256:', sha256_file(INDEX_ARCHIVE))
print('File count:', len(all_files))
print('Extension counts:', Counter(p.suffix.lower() for p in all_files))
print('Index 003 subdirectories:')
for path in sorted(p for p in INDEX_ROOT.rglob('*') if p.is_dir()):
    print(' ', path.relative_to(INDEX_ROOT))

metadata_files = [p for p in all_files if p.suffix.lower() in ('.txt', '.md', '.xml', '.json', '.csv', '.yaml', '.yml')]
for path in metadata_files[:80]:
    print('METADATA', path, 'sha256=', sha256_file(path))

label_candidates = [p for p in all_files if 'groundtruth' in p.name.lower()
                    and p.suffix.lower() == '.png']
assert label_candidates, 'No GroundTruth file under dataset 003; inspect archive layout before proceeding'
label_samples = []
for path in label_candidates[:10]:
    with Image.open(path) as label_image:
        print('LABEL', path, 'mode=', label_image.mode, 'size=', label_image.size,
              'bands=', label_image.getbands(), 'format=', label_image.format)
        print('  palette/info keys:', sorted(label_image.info.keys()))
        label_samples.append({'path': str(path), 'mode': label_image.mode,
                              'size': list(label_image.size), 'bands': list(label_image.getbands()),
                              'format': label_image.format,
                              'info_keys': sorted(label_image.info.keys())})

tile_root = INDEX_ROOT / '003' / 'tile'
assert tile_root.is_dir(), f'Index 003 tile root not found: {tile_root}'
channel_dirs = [tile_root / band for band in ('R', 'G', 'B', 'NIR', 'RE')]
missing_channels = [str(path) for path in channel_dirs if not path.is_dir()]
assert not missing_channels, f'Missing required RedEdge channels: {missing_channels}'
rededge_candidates = [path for channel in channel_dirs for path in channel.glob('*.png')]
print('Index 003 per-band tile counts:', {path.name: len(list(path.glob('*.png')))
                                        for path in channel_dirs})
archive_hashes = {str(INDEX_ARCHIVE): sha256_file(INDEX_ARCHIVE)}
inventory = {
    'source_page': 'https://projects.asl.ethz.ch/datasets/weedmap-2018/',
    'paper_doi': '10.3390/rs10091423',
    'dataset_doi': '10.3929/ethz-c-000788571',
    'dataset_root': str(WM_ROOT),
    'dataset_index_root': str(INDEX_ROOT),
    'file_count': len(all_files),
    'dataset_003_rededge_candidate_count': len(rededge_candidates),
    'rededge_sample_paths': [str(path) for path in rededge_candidates[:100]],
    'label_samples': label_samples,
    'metadata_files': [str(path) for path in metadata_files],
    'archive_hashes': archive_hashes,
    'dataset_identity': {
        'dataset_index': '003',
        'sensor': 'MicaSense RedEdge-M',
        'tile_archive': str(INDEX_ARCHIVE),
        'tile_archive_sha256': sha256_file(INDEX_ARCHIVE),
    },
    'mask_validity_policy': 'PENDING: release says white is valid; inspected mask/signal relation appeared opposite',
    'verified_class_map': {'0': 'background', '1': 'crop', '2': 'weed', '10000': 'ignore'},
    'verified_label_semantics': {
        'verified': True,
        'codes': {'0': 'background', '1': 'crop', '2': 'weed', '10000': 'non-class_ignore'},
        'source': 'ETH Zurich Research Collection WeedMap release PDF, page 2',
        'source_url': 'https://www.research-collection.ethz.ch/server/api/core/bitstreams/3ae97814-1570-4760-95b8-a9a3eac21975/content',
    },
    'verified_mask_validity': None,
    'verified_class_support': None,
    'verified_input_transform': None,
    'inventory_status': 'STRUCTURE_ONLY_NOT_AUTHORIZATION',
    'source_test_split_citation': None,
    'verified_grouping': None,
    'manual_review_fields': ['official partition 003 and source citation',
                             'five co-registered RedEdge-M bands and channel order',
                             'radiometric calibration/PNG encoding and fixed source-compatible transform',
                             'mask validity polarity resolved against release',
                             'indexed semantics verified; class support and instance availability documented',
                             'license/reuse conditions and tile/orthomosaic grouping reviewed'],
    'preflight_note': 'Inventory only. Copy to a separate reviewed preflight artifact; do not authorize from this file.',
}
inventory_path = f'{OUT}/weedmap_003_inventory.json'
with open(inventory_path, 'w') as f:
    json.dump(inventory, f, indent=2)
print('Saved inventory:', inventory_path)
print('Existing reviewed preflight was not modified.')
print('Inventory only. Do not evaluate model predictions or tune from these labels.')

Mounted root: /kaggle/working/weedmap-2018
Dataset index root: /kaggle/working/weedmap-2018/Tiles/RedEdge/003
Index 003 archive sha256: f3a5ffcca6921ce146bb3ee92235142bbb16a96375ea81ee9a744128094e7f5d
File count: 2455
Extension counts: Counter({'.png': 2448, '': 4, '.ds_store': 3})
Index 003 subdirectories:
  003
  003/groundtruth
  003/mask
  003/tile
  003/tile/B
  003/tile/CIR
  003/tile/G
  003/tile/NDVI
  003/tile/NIR
  003/tile/R
  003/tile/RE
  003/tile/RGB
  __MACOSX
  __MACOSX/003
  __MACOSX/003/mask
  __MACOSX/003/tile
LABEL /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/groundtruth/003_frame0021_GroundTruth_iMap.png mode= I;16 size= (480, 360) bands= ('I',) format= PNG
  palette/info keys: []
LABEL /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/groundtruth/003_frame0052_GroundTruth_iMap.png mode= I;16 size= (480, 360) bands= ('I',) format= PNG
  palette/info keys: []
LABEL /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/groundtruth/003_frame0131_GroundTruth_color.

In [20]:
#Cell K8.0g — Verify pushed commit and checkpoint artifact presence (no test data access)
import hashlib, json, os, subprocess
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
EXPECTED_COMMIT = 'fe3daf1a91203b20d73aac960d08f2dbf96be872'
assert PROJECT.is_dir(), f'Project not found: {PROJECT}'
os.chdir(PROJECT)
OUT = Path('results/exp08')
OUT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Kaggle HEAD:', actual)
print('Expected   :', EXPECTED_COMMIT)
assert actual == EXPECTED_COMMIT, 'Kaggle has the wrong commit; pull/clone the pushed source before continuing'
assert not (OUT / 'test_started.json').exists(), 'A test marker exists: STOP and review previous run'

paths = [Path(f'runs/{method}_s{seed}/{name}')
         for method in ('L0', 'OccCondg', 'MCSDg')
         for seed in range(5)
         for name in ('best_val.pth', 'run_config.json')]
paths += [Path('src/weedmap_evaluation.py'), Path('tests/test_weedmap_evaluation.py')]
records = {str(p): {'exists': p.is_file(), 'size_bytes': p.stat().st_size if p.is_file() else None,
                    'sha256': sha256(p) if p.is_file() else None} for p in paths}
missing = [p for p, data in records.items() if not data['exists']]
report = {'status': 'AUDIT_ONLY_NOT_TEST_AUTHORIZATION', 'utc': datetime.now(timezone.utc).isoformat(),
          'git_commit': actual, 'expected_commit': EXPECTED_COMMIT, 'artifacts': records,
          'missing': missing, 'checkpoint_pairs_expected': 15, 'test_data_opened': False}
dest = OUT / 'checkpoint_source_presence_audit.json'
dest.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Expected checkpoint/config pairs: 15')
print('Missing paths:', len(missing), missing[:20])
print('Saved:', dest)
print('No test data or checkpoint inference was opened.')

Kaggle HEAD: cb804fd6e4024405c5538239d71029b2eeabe430
Expected   : fe3daf1a91203b20d73aac960d08f2dbf96be872


AssertionError: Kaggle has the wrong commit; pull/clone the pushed source before continuing

In [16]:
#Cell K8.0d — Full-index class support and mask-polarity integrity audit (no inference)
import json
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image

ROOT = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
OUT = Path('results/exp08')
LOG = OUT / 'logs' / 'class_support_mask_integrity.log'
REPORT = OUT / 'weedmap_003_class_support_integrity.json'
OUT.mkdir(parents=True, exist_ok=True)
LOG.parent.mkdir(parents=True, exist_ok=True)
BANDS = ('R', 'G', 'B', 'NIR', 'RE')
CODE_NAMES = {0: 'background', 1: 'crop', 2: 'weed', 10000: 'non_class_ignore'}
frames = sorted(path.stem for path in (ROOT / 'tile' / 'R').glob('frame*.png'))
assert len(frames) == 204, f'Expected 204 R frames in index 003, found {len(frames)}'

all_counts = Counter()
counts_by_mask = {0: Counter(), 255: Counter()}
mask_patterns = Counter()
imap_modes = Counter()
missing_pairs = []
shape_mismatches = []
frame_rows = []
band_frame_sets = {
    band: {path.stem for path in (ROOT / 'tile' / band).glob('frame*.png')}
    for band in BANDS
}
for band, band_frames in band_frame_sets.items():
    assert band_frames == set(frames), f'{band} frame IDs differ from R'

with LOG.open('w', encoding='utf-8', buffering=1) as log:
    def emit(message):
        print(message, flush=True)
        log.write(message + '\n')

    emit('WeedMap 003 class-support/mask integrity audit; no model inference.')
    emit('Counts are reported separately for mask==0 and mask==255; neither is presumed valid.')
    for frame in frames:
        imap_path = ROOT / 'groundtruth' / f'003_{frame}_GroundTruth_iMap.png'
        mask_path = ROOT / 'mask' / f'{frame}.png'
        if not imap_path.is_file() or not mask_path.is_file():
            missing_pairs.append(frame)
            continue
        with Image.open(imap_path) as image:
            imap = np.asarray(image)
            imap_mode = image.mode
        with Image.open(mask_path) as image:
            mask = np.asarray(image)
        if imap.ndim != 2 or mask.ndim != 2 or imap.shape != mask.shape:
            shape_mismatches.append({'frame': frame, 'imap_shape': list(imap.shape),
                                     'mask_shape': list(mask.shape)})
            continue
        unknown = set(np.unique(imap).tolist()) - set(CODE_NAMES)
        assert not unknown, f'Unexpected iMap codes in {frame}: {sorted(unknown)}'
        imap_modes[imap_mode] += 1
        mask_values, mask_counts = np.unique(mask, return_counts=True)
        mask_patterns[tuple(int(value) for value in mask_values)] += 1
        frame_counts = Counter({int(code): int(count) for code, count in
                                zip(*np.unique(imap, return_counts=True))})
        all_counts.update(frame_counts)
        for mask_value in (0, 255):
            values, counts = np.unique(imap[mask == mask_value], return_counts=True)
            counts_by_mask[mask_value].update({int(code): int(count) for code, count in zip(values, counts)})
        frame_rows.append({'frame': frame, 'imap_mode': imap_mode,
                           'mask_counts': {str(int(value)): int(count) for value, count
                                           in zip(mask_values, mask_counts)},
                           'class_counts': {str(code): frame_counts.get(code, 0) for code in CODE_NAMES}})

    assert not missing_pairs, f'Missing iMap/mask pairs: {missing_pairs}'
    assert not shape_mismatches, f'iMap/mask shape mismatches: {shape_mismatches[:10]}'
    crop_count = all_counts.get(1, 0)
    result = {
        'dataset_index': '003',
        'tile_archive': '/kaggle/working/weedmap-2018/Tiles/RedEdge/003.zip',
        'frames_expected': len(frames),
        'frames_aligned': len(frame_rows),
        'band_frame_counts': {band: len(ids) for band, ids in band_frame_sets.items()},
        'imap_modes': dict(imap_modes),
        'mask_patterns_by_frame': {str(key): value for key, value in mask_patterns.items()},
        'class_codes': {str(code): name for code, name in CODE_NAMES.items()},
        'all_imap_counts': {str(code): int(all_counts.get(code, 0)) for code in CODE_NAMES},
        'counts_if_mask_zero_valid': {str(code): int(counts_by_mask[0].get(code, 0)) for code in CODE_NAMES},
        'counts_if_mask_255_valid': {str(code): int(counts_by_mask[255].get(code, 0)) for code in CODE_NAMES},
        'crop_code_present_in_any_imap_pixel': bool(crop_count),
        'mask_validity_decision': 'PENDING; both hypotheses retained',
        'inference_run': False,
        'per_frame': frame_rows,
    }
    with REPORT.open('w', encoding='utf-8') as stream:
        json.dump(result, stream, indent=2)
    emit(f'Frames aligned: {len(frame_rows)} / {len(frames)}')
    emit(f'iMap modes: {dict(imap_modes)}')
    emit(f'Mask patterns by frame: {dict(mask_patterns)}')
    emit(f'All iMap class counts: {result["all_imap_counts"]}')
    emit(f'Counts if mask==0 is valid: {result["counts_if_mask_zero_valid"]}')
    emit(f'Counts if mask==255 is valid: {result["counts_if_mask_255_valid"]}')
    emit(f'Crop code 1 present anywhere: {bool(crop_count)}')
    emit('No mask polarity selected; no model inference performed.')
    emit(f'Saved report: {REPORT}')
    emit(f'Saved log: {LOG}')

WeedMap 003 class-support/mask integrity audit; no model inference.
Counts are reported separately for mask==0 and mask==255; neither is presumed valid.
Frames aligned: 204 / 204
iMap modes: {'L': 111, 'I;16': 93}
Mask patterns by frame: {(255,): 102, (0, 255): 57, (0,): 45}
All iMap class counts: {'0': 34239045, '1': 0, '2': 343926, '10000': 668229}
Counts if mask==0 is valid: {'0': 12000413, '1': 0, '2': 343799, '10000': 668227}
Counts if mask==255 is valid: {'0': 22238632, '1': 0, '2': 127, '10000': 2}
Crop code 1 present anywhere: False
No mask polarity selected; no model inference performed.
Saved report: results/exp08/weedmap_003_class_support_integrity.json
Saved log: results/exp08/logs/class_support_mask_integrity.log


In [17]:
#Cell K8.0e — Run synthetic WeedMap loader/metric tests on Kaggle (no real dataset/inference)
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
assert PROJECT.is_dir(), f'Project not found: {PROJECT}'
os.chdir(PROJECT)
source_path = Path('src/weedmap_evaluation.py')
test_path = Path('tests/test_weedmap_evaluation.py')
assert source_path.is_file() and test_path.is_file(), 'Push/clone both revised Python files first'

# Prevent accidentally testing the older metric-only files.
source_text = source_path.read_text(encoding='utf-8')
test_text = test_path.read_text(encoding='utf-8')
for symbol in ('def discover_frames(', 'def load_tile(', 'def resolve_valid_pixels(',
               'def aggregate_predictions('):
    assert symbol in source_text, f'Expected revised evaluator primitive missing: {symbol}'
assert test_text.count('def test_') >= 14, 'Expected revised synthetic test suite (14 tests)'

OUT = Path('results/exp08')
LOGS = OUT / 'logs'
LOGS.mkdir(parents=True, exist_ok=True)
log_path = LOGS / 'weedmap_synthetic_core_tests.log'
report_path = OUT / 'weedmap_synthetic_core_test_report.json'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

command = [sys.executable, '-m', 'pytest', '-q', '-ra', str(test_path)]
print('$', ' '.join(command), flush=True)
with log_path.open('w', encoding='utf-8', buffering=1) as log:
    proc = subprocess.Popen(command, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    exit_code = proc.wait()

report = {
    'status': 'SYNTHETIC_CORE_PASS' if exit_code == 0 else 'SYNTHETIC_CORE_FAIL',
    'authorization': 'NOT_AUTHORIZED_FOR_WEEDMAP_INFERENCE_OR_K8_2_K8_3',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'command': command,
    'pytest_exit_code': exit_code,
    'source_file': str(source_path),
    'source_sha256': sha256(source_path),
    'test_file': str(test_path),
    'test_sha256': sha256(test_path),
    'log_file': str(log_path),
    'log_sha256': sha256(log_path),
    'scope': 'Synthetic loader, band order, mask-code handling, labels, confusion and aggregation only',
    'not_verified': ['radiometry', 'official mask polarity', 'official test split',
                     'spatial grouping', 'checkpoint inference integration'],
}
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved synthetic test report:', report_path, flush=True)
print('Saved pytest log:', log_path, flush=True)
assert exit_code == 0, 'Synthetic tests failed: inspect the saved pytest log'
print('Synthetic code tests PASS; K8.2/K8.3 remain NO-GO.', flush=True)

$ /usr/bin/python3 -m pytest -q -ra tests/test_weedmap_evaluation.py
..............                                                           [100%]
14 passed in 0.17s
Saved synthetic test report: results/exp08/weedmap_synthetic_core_test_report.json
Saved pytest log: results/exp08/logs/weedmap_synthetic_core_tests.log
Synthetic code tests PASS; K8.2/K8.3 remain NO-GO.


In [18]:
#Cell K8.0f — Backup Exp08 audit/test artifacts (no inference)
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
exp08 = PROJECT / 'results' / 'exp08'
backup = Path('/kaggle/working/exp08_preflight_synthetic_backup.zip')
assert exp08.is_dir(), f'Exp08 outputs not found: {exp08}'
with ZipFile(backup, 'w', compression=ZIP_DEFLATED) as archive:
    for path in sorted(exp08.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(PROJECT)))
print('Saved:', backup)
print('Size MB:', round(backup.stat().st_size / 1024**2, 2))
from IPython.display import FileLink, display
display(FileLink(str(backup)))

Saved: /kaggle/working/exp08_preflight_synthetic_backup.zip
Size MB: 0.01


/kaggle/working/exp08_preflight_synthetic_backup.zip

In [14]:
import os

for root, dirs, files in os.walk(WM_ROOT):
    level = root.replace(WM_ROOT, '').count(os.sep)

    if level <= 3:
        print('  ' * level + os.path.basename(root) + '/')

        for filename in sorted(files)[:10]:
            print('  ' * (level + 1) + filename)


weedmap-2018/
  Tiles/
    RedEdge.zip
    Sequoia.zip
    Sequoia/
      005.zip
      006.zip
      007.zip
    RedEdge/
      000.zip
      001.zip
      002.zip
      003.zip
      004.zip
      000/
        .DS_Store


In [15]:
import os
import zipfile

WM_ROOT = '/kaggle/working/weedmap-2018'

ARCHIVE = os.path.join(
    WM_ROOT,
    'Tiles',
    'RedEdge',
    '003.zip'
)

OUTPUT = os.path.join(
    WM_ROOT,
    'Tiles',
    'RedEdge',
    '003'
)

assert os.path.isfile(ARCHIVE), (
    f'Missing RedEdge partition 003: {ARCHIVE}'
)

os.makedirs(OUTPUT, exist_ok=True)

with zipfile.ZipFile(ARCHIVE, 'r') as z:
    print('Archive members:', len(z.namelist()))

    bad = z.testzip()
    assert bad is None, f'Corrupt archive member: {bad}'

    z.extractall(OUTPUT)

print('Extracted successfully:', OUTPUT)


Archive members: 2471
Extracted successfully: /kaggle/working/weedmap-2018/Tiles/RedEdge/003


In [29]:
#kiểm tra môi trường này trước:
from pathlib import Path
import os

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
WM = Path('/kaggle/working/weedmap-2018')

checks = {
    'Project': PROJECT.is_dir(),
    'WeedMap 003 archive':
        (WM / 'Tiles/RedEdge/003.zip').is_file(),
    'WeedMap 003 extracted':
        (WM / 'Tiles/RedEdge/003/003/tile/R').is_dir(),
    'WeedMap frame0020':
        (WM / 'Tiles/RedEdge/003/003/tile/R/frame0020.png').is_file(),
}

for name, passed in checks.items():
    print(f'{name}: {"OK" if passed else "MISSING"}')

assert all(checks.values()), 'Environment is not ready'
os.chdir(PROJECT)

print('\nReady for K8.0, K8.0b and K8.0c')


Project: OK
WeedMap 003 archive: OK
WeedMap 003 extracted: OK
WeedMap frame0020: OK

Ready for K8.0, K8.0b and K8.0c


In [30]:
#Cell K8.0 — WeedMap RedEdge-M 003 inventory (no inference)
import glob, hashlib, json, os, pathlib, sys
from collections import Counter
from PIL import Image

PROJECT = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(PROJECT)
EXP = 'exp08'
OUT = f'results/{EXP}'
os.makedirs(f'{OUT}/logs', exist_ok=True)

WM_CANDIDATES = ['/kaggle/working/weedmap-2018'] + glob.glob(
    '/kaggle/input/**/weedmap-2018', recursive=True)
WM_ROOT = next((path for path in WM_CANDIDATES if os.path.isdir(path)), None)
assert WM_ROOT, f'WeedMap root not found: {WM_CANDIDATES}'
WM_ROOT = pathlib.Path(WM_ROOT)
INDEX_ROOT = WM_ROOT / 'Tiles' / 'RedEdge' / '003'
INDEX_ARCHIVE = WM_ROOT / 'Tiles' / 'RedEdge' / '003.zip'
assert INDEX_ROOT.is_dir(), f'Extracted RedEdge index 003 not found: {INDEX_ROOT}'
assert INDEX_ARCHIVE.is_file(), f'RedEdge index 003 archive not found: {INDEX_ARCHIVE}'

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

all_files = [p for p in INDEX_ROOT.rglob('*') if p.is_file()]
print('Mounted root:', WM_ROOT)
print('Dataset index root:', INDEX_ROOT)
print('Index 003 archive sha256:', sha256_file(INDEX_ARCHIVE))
print('File count:', len(all_files))
print('Extension counts:', Counter(p.suffix.lower() for p in all_files))
print('Index 003 subdirectories:')
for path in sorted(p for p in INDEX_ROOT.rglob('*') if p.is_dir()):
    print(' ', path.relative_to(INDEX_ROOT))

metadata_files = [p for p in all_files if p.suffix.lower() in ('.txt', '.md', '.xml', '.json', '.csv', '.yaml', '.yml')]
for path in metadata_files[:80]:
    print('METADATA', path, 'sha256=', sha256_file(path))

label_candidates = [p for p in all_files if 'groundtruth' in p.name.lower()
                    and p.suffix.lower() == '.png']
assert label_candidates, 'No GroundTruth file under dataset 003; inspect archive layout before proceeding'
label_samples = []
for path in label_candidates[:10]:
    with Image.open(path) as label_image:
        print('LABEL', path, 'mode=', label_image.mode, 'size=', label_image.size,
              'bands=', label_image.getbands(), 'format=', label_image.format)
        print('  palette/info keys:', sorted(label_image.info.keys()))
        label_samples.append({'path': str(path), 'mode': label_image.mode,
                              'size': list(label_image.size), 'bands': list(label_image.getbands()),
                              'format': label_image.format,
                              'info_keys': sorted(label_image.info.keys())})

tile_root = INDEX_ROOT / '003' / 'tile'
assert tile_root.is_dir(), f'Index 003 tile root not found: {tile_root}'
channel_dirs = [tile_root / band for band in ('R', 'G', 'B', 'NIR', 'RE')]
missing_channels = [str(path) for path in channel_dirs if not path.is_dir()]
assert not missing_channels, f'Missing required RedEdge channels: {missing_channels}'
rededge_candidates = [path for channel in channel_dirs for path in channel.glob('*.png')]
print('Index 003 per-band tile counts:', {path.name: len(list(path.glob('*.png')))
                                        for path in channel_dirs})
archive_hashes = {str(INDEX_ARCHIVE): sha256_file(INDEX_ARCHIVE)}
inventory = {
    'source_page': 'https://projects.asl.ethz.ch/datasets/weedmap-2018/',
    'paper_doi': '10.3390/rs10091423',
    'dataset_doi': '10.3929/ethz-c-000788571',
    'dataset_root': str(WM_ROOT),
    'dataset_index_root': str(INDEX_ROOT),
    'file_count': len(all_files),
    'dataset_003_rededge_candidate_count': len(rededge_candidates),
    'rededge_sample_paths': [str(path) for path in rededge_candidates[:100]],
    'label_samples': label_samples,
    'metadata_files': [str(path) for path in metadata_files],
    'archive_hashes': archive_hashes,
    'dataset_identity': {
        'dataset_index': '003',
        'sensor': 'MicaSense RedEdge-M',
        'tile_archive': str(INDEX_ARCHIVE),
        'tile_archive_sha256': sha256_file(INDEX_ARCHIVE),
    },
    'mask_validity_policy': 'PENDING: release says white is valid; inspected mask/signal relation appeared opposite',
    'verified_class_map': {'0': 'background', '1': 'crop', '2': 'weed', '10000': 'ignore'},
    'verified_label_semantics': {
        'verified': True,
        'codes': {'0': 'background', '1': 'crop', '2': 'weed', '10000': 'non-class_ignore'},
        'source': 'ETH Zurich Research Collection WeedMap release PDF, page 2',
        'source_url': 'https://www.research-collection.ethz.ch/server/api/core/bitstreams/3ae97814-1570-4760-95b8-a9a3eac21975/content',
    },
    'verified_mask_validity': None,
    'verified_class_support': None,
    'verified_input_transform': None,
    'inventory_status': 'STRUCTURE_ONLY_NOT_AUTHORIZATION',
    'source_test_split_citation': None,
    'verified_grouping': None,
    'manual_review_fields': ['official partition 003 and source citation',
                             'five co-registered RedEdge-M bands and channel order',
                             'radiometric calibration/PNG encoding and fixed source-compatible transform',
                             'mask validity polarity resolved against release',
                             'indexed semantics verified; class support and instance availability documented',
                             'license/reuse conditions and tile/orthomosaic grouping reviewed'],
    'preflight_note': 'Inventory only. Copy to a separate reviewed preflight artifact; do not authorize from this file.',
}
inventory_path = f'{OUT}/weedmap_003_inventory.json'
with open(inventory_path, 'w') as f:
    json.dump(inventory, f, indent=2)
print('Saved inventory:', inventory_path)
print('Existing reviewed preflight was not modified.')
print('Inventory only. Do not evaluate model predictions or tune from these labels.')

Mounted root: /kaggle/working/weedmap-2018
Dataset index root: /kaggle/working/weedmap-2018/Tiles/RedEdge/003
Index 003 archive sha256: f3a5ffcca6921ce146bb3ee92235142bbb16a96375ea81ee9a744128094e7f5d
File count: 2455
Extension counts: Counter({'.png': 2448, '': 4, '.ds_store': 3})
Index 003 subdirectories:
  003
  003/groundtruth
  003/mask
  003/tile
  003/tile/B
  003/tile/CIR
  003/tile/G
  003/tile/NDVI
  003/tile/NIR
  003/tile/R
  003/tile/RE
  003/tile/RGB
  __MACOSX
  __MACOSX/003
  __MACOSX/003/mask
  __MACOSX/003/tile
LABEL /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/groundtruth/003_frame0139_GroundTruth_iMap.png mode= L size= (480, 360) bands= ('L',) format= PNG
  palette/info keys: []
LABEL /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/groundtruth/003_frame0033_GroundTruth_iMap.png mode= I;16 size= (480, 360) bands= ('I',) format= PNG
  palette/info keys: []
LABEL /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/groundtruth/003_frame0005_GroundTruth_color.png

In [31]:
#Cell K8.0b — Read WeedMap annotation documentation/source only (no label pixels)
import hashlib
import os
import pathlib
import re
import glob
import zipfile

root_candidates = [pathlib.Path('/kaggle/working/weedmap-2018')]
root_candidates += [pathlib.Path(path) for path in glob.glob('/kaggle/input/**/weedmap-2018', recursive=True)]
WM_ROOT = next((path for path in root_candidates if path.is_dir()), None)
AUDIT_LOG = pathlib.Path('results/exp08/logs/weedmap_annotation_docs_audit.log')
AUDIT_LOG.parent.mkdir(parents=True, exist_ok=True)
assert WM_ROOT is not None, f'WeedMap root not found in candidates: {root_candidates}'

text_extensions = {
    '.txt', '.md', '.rst', '.html', '.htm', '.xml', '.json', '.yaml', '.yml',
    '.csv', '.m', '.py', '.tex', '.ini', '.cfg'
}
search_pattern = re.compile(
    r'10000|ground.?truth.?imap|ground.?truth.?color|background|crop|weed|'
    r'ignore|unlabel|non.?class|annotation|label.?map|class.?code',
    re.IGNORECASE,
)
exact_code_pattern = re.compile(r'(?<!\d)10000(?!\d)|GroundTruth[_ -]?(?:iMap|color)', re.IGNORECASE)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def emit(message=''):
    print(message, flush=True)
    log.write(message + '\n')

text_files = sorted(
    path for path in WM_ROOT.rglob('*')
    if path.is_file() and path.suffix.lower() in text_extensions
    and path.stat().st_size <= 4 * 1024 * 1024
)
zip_candidates = sorted({
    path for path in [WM_ROOT / 'Tiles' / 'RedEdge' / '003.zip'] + list(WM_ROOT.rglob('003.zip'))
    if path.is_file() and 'rededge' in str(path).lower()
})
exact_hits = 0
files_with_hits = 0
with AUDIT_LOG.open('w', encoding='utf-8', buffering=1) as log:
    emit('WeedMap annotation documentation/source audit')
    emit(f'Root: {WM_ROOT}')
    emit('Scope: text files only; no PNG/TIFF/mask pixels opened.')
    emit(f'Text files scanned: {len(text_files)} (max 4 MiB each)')
    sources = []
    for path in text_files:
        try:
            sources.append((str(path.relative_to(WM_ROOT)), path.read_text(
                encoding='utf-8-sig', errors='replace').splitlines(), sha256_file(path)))
        except OSError as error:
            emit(f'SKIP {path.relative_to(WM_ROOT)}: {error}')
    for archive in zip_candidates:
        emit(f'ZIP {archive.relative_to(WM_ROOT)} sha256={sha256_file(archive)}')
        try:
            with zipfile.ZipFile(archive) as bundle:
                members = [item for item in bundle.infolist()
                           if pathlib.PurePosixPath(item.filename).suffix.lower() in text_extensions
                           and 0 < item.file_size <= 4 * 1024 * 1024]
                emit(f'  text members eligible: {len(members)}')
                for item in members:
                    with bundle.open(item) as stream:
                        content = stream.read(4 * 1024 * 1024 + 1)
                    if len(content) > 4 * 1024 * 1024:
                        continue
                    lines = content.decode('utf-8-sig', errors='replace').splitlines()
                    sources.append((f'{archive.relative_to(WM_ROOT)}::{item.filename}', lines,
                                    hashlib.sha256(content).hexdigest()))
        except (OSError, zipfile.BadZipFile) as error:
            emit(f'  ZIP READ ERROR: {error}')
    for source_name, lines, source_hash in sources:
        matches = [(index, line.strip()) for index, line in enumerate(lines, start=1)
                   if search_pattern.search(line)]
        exact_matches = [(index, line) for index, line in matches
                         if exact_code_pattern.search(line)]
        if not matches:
            continue
        files_with_hits += 1
        exact_hits += len(exact_matches)
        emit(f'\nTEXT SOURCE {source_name} sha256={source_hash}')
        selected = exact_matches if exact_matches else matches[:5]
        if exact_matches:
            context_lines = set()
            for index, _ in exact_matches:
                context_lines.update(range(max(1, index - 2), min(len(lines), index + 2) + 1))
            selected = [(index, lines[index - 1].strip()) for index in sorted(context_lines)]
        else:
            selected = matches[:5]
        for index, line in selected[:20]:
            emit(f'  L{index}: {line[:500]}')
    emit('\nARCHIVE TEXT SEARCH: COMPLETE; this is supplementary and does not determine label semantics.')
    emit('Official semantics remain those in the cited ETH Zurich release PDF, page 2.')
    emit(f'Files with relevant text: {files_with_hits}; exact code/name hits: {exact_hits}')
    emit(f'Saved audit: {AUDIT_LOG}')
print('Saved audit:', AUDIT_LOG, flush=True)

WeedMap annotation documentation/source audit
Root: /kaggle/working/weedmap-2018
Scope: text files only; no PNG/TIFF/mask pixels opened.
Text files scanned: 0 (max 4 MiB each)
ZIP Tiles/RedEdge/003.zip sha256=f3a5ffcca6921ce146bb3ee92235142bbb16a96375ea81ee9a744128094e7f5d
  text members eligible: 0

ARCHIVE TEXT SEARCH: COMPLETE; this is supplementary and does not determine label semantics.
Official semantics remain those in the cited ETH Zurich release PDF, page 2.
Files with relevant text: 0; exact code/name hits: 0
Saved audit: results/exp08/logs/weedmap_annotation_docs_audit.log
Saved audit: results/exp08/logs/weedmap_annotation_docs_audit.log


In [32]:
#Cell K8.0c — Input encoding/range inspection (imagery only; no transform fitting)
import glob
import os
from pathlib import Path

import matplotlib.image as mpimg
import numpy as np
from PIL import Image

PROJECT = Path('/kaggle/working/WeedsGalore_Ipr28')
os.chdir(PROJECT)
AUDIT_LOG = Path('results/exp08/logs/radiometry_encoding_audit.log')
AUDIT_LOG.parent.mkdir(parents=True, exist_ok=True)
WM_CANDIDATES = [Path('/kaggle/working/weedmap-2018')]
WM_CANDIDATES += [Path(path) for path in glob.glob('/kaggle/input/**/weedmap-2018', recursive=True)]
WM_ROOT = next((path for path in WM_CANDIDATES if path.is_dir()), None)
assert WM_ROOT is not None, f'Cannot find WeedMap root: {WM_CANDIDATES}'
WG_CANDIDATES = sorted(glob.glob('/kaggle/input/**/2023-05-25/images', recursive=True))
if WG_CANDIDATES:
    WG_ROOT = Path(WG_CANDIDATES[0]).parent.parent
elif Path('/kaggle/working/weedsgalore-dataset/2023-05-25/images').is_dir():
    WG_ROOT = Path('/kaggle/working/weedsgalore-dataset')
else:
    raise FileNotFoundError('Cannot find WeedsGalore images under Kaggle input/working')

WM_TILE_ROOT = WM_ROOT / 'Tiles' / 'RedEdge' / '003' / '003' / 'tile'
assert WM_TILE_ROOT.is_dir(), f'WeedMap tile root missing: {WM_TILE_ROOT}'
bands = ('R', 'G', 'B', 'NIR', 'RE')
wg_files = sorted((WG_ROOT / '2023-05-25' / 'images').glob('*_R.png'))
assert wg_files, f'No WeedsGalore sample images under {WG_ROOT}'
wg_stem = wg_files[0].name[:-len('_R.png')]
wm_stem = 'frame0020'
assert (WM_TILE_ROOT / 'R' / f'{wm_stem}.png').is_file(), (
    f'Predeclared WeedMap radiometry sample is missing: {wm_stem}')

def report_image(domain, band, path):
    with Image.open(path) as image:
        raw = np.asarray(image)
        mode = image.mode
        size = image.size
    if domain == 'WeedMap-003-RedEdge':
        assert size == (480, 360), f'Unexpected predeclared frame size: {path} {size}'
        assert int(raw.max()) > 0, f'Predeclared frame is blank in {band}: {path}'
    loaded = mpimg.imread(path)
    finite = loaded[np.isfinite(loaded)]
    quantiles = np.percentile(finite, [0, 1, 50, 99, 100])
    record = {
        'domain': domain,
        'band': band,
        'path': str(path),
        'png_mode': mode,
        'raw_dtype': str(raw.dtype),
        'raw_shape': list(raw.shape),
        'size_wh': list(size),
        'raw_min_max': [int(raw.min()), int(raw.max())],
        'plt_imread_dtype': str(loaded.dtype),
        'plt_imread_min_p01_p50_p99_max': [float(v) for v in quantiles],
    }
    print(record, flush=True)
    audit.write(str(record) + '\n')

with AUDIT_LOG.open('w', encoding='utf-8', buffering=1) as audit:
    print('WeedsGalore root:', WG_ROOT, '| sample:', wg_stem, flush=True)
    print('WeedMap root:', WM_TILE_ROOT, '| sample:', wm_stem, flush=True)
    audit.write(f'WeedsGalore root: {WG_ROOT} | sample: {wg_stem}\n')
    audit.write(f'WeedMap root: {WM_TILE_ROOT} | sample: {wm_stem}\n')
    for band in bands:
        wg_path = WG_ROOT / '2023-05-25' / 'images' / f'{wg_stem}_{band}.png'
        wm_path = WM_TILE_ROOT / band / f'{wm_stem}.png'
        assert wg_path.is_file(), f'Missing WeedsGalore {band}: {wg_path}'
        assert wm_path.is_file(), f'Missing WeedMap {band}: {wm_path}'
        with Image.open(wm_path) as image:
            assert image.size == (480, 360), f'Unexpected WeedMap frame0020 size for {band}: {image.size}'
        report_image('WeedsGalore', band, wg_path)
        report_image('WeedMap-003-RedEdge', band, wm_path)
    print('No labels/masks were opened. Do not use these summaries to fit or tune a transform.', flush=True)
    print('Radiometric compatibility remains PENDING until source calibration/encoding documentation is verified.', flush=True)
    audit.write('No labels/masks were opened. No transform fitted. Radiometry remains PENDING.\n')
print('Saved audit:', AUDIT_LOG, flush=True)

WeedsGalore root: /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset | sample: 2023-05-25_0109
WeedMap root: /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/tile | sample: frame0020
{'domain': 'WeedsGalore', 'band': 'R', 'path': '/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/2023-05-25_0109_R.png', 'png_mode': 'I;16', 'raw_dtype': 'uint16', 'raw_shape': [600, 600], 'size_wh': [600, 600], 'raw_min_max': [7744, 64576], 'plt_imread_dtype': 'float32', 'plt_imread_min_p01_p50_p99_max': [0.11816586554050446, 0.2080109864473343, 0.4746166169643402, 0.7099717855453491, 0.9853665828704834]}
{'domain': 'WeedMap-003-RedEdge', 'band': 'R', 'path': '/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003/tile/R/frame0020.png', 'png_mode': 'L', 'raw_dtype': 'uint8', 'raw_shape': [360, 480], 'size_wh': [480, 360], 'raw_min_max': [23, 195], 'plt_imread_dtype': 'float32', 'plt_imread_min_p01_p50_p99_max': [0.09019608050584793, 0.2000000

In [33]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

project = Path('/kaggle/working/WeedsGalore_Ipr28')
results = project / 'results'
output = Path('/kaggle/working/exp08_results_backup.zip')

assert results.is_dir(), f'Not found: {results}'

with ZipFile(output, 'w', compression=ZIP_DEFLATED) as z:
    for file in sorted((results / 'exp08').rglob('*')):
        if file.is_file():
            z.write(file, arcname=str(file.relative_to(project)))

print('Saved:', output)
print('Size:', round(output.stat().st_size / (1024**2), 2), 'MB')

Saved: /kaggle/working/exp08_results_backup.zip
Size: 0.01 MB


In [17]:
#Cell K8.0 — WeedMap RedEdge-M 003 preflight (no inference)
import glob, hashlib, json, os, pathlib, sys
from collections import Counter
from PIL import Image

PROJECT = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(PROJECT)
EXP = 'exp08'
OUT = f'results/{EXP}'
os.makedirs(f'{OUT}/logs', exist_ok=True)

# Set this to the mounted archive's extracted root; do not guess a Sequoia directory.
WM_ROOT = '/kaggle/working/weedmap-2018'
assert os.path.isdir(WM_ROOT), f'WeedMap root not found: {WM_ROOT}'

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

all_files = [p for p in pathlib.Path(WM_ROOT).rglob('*') if p.is_file()]
print('Mounted root:', WM_ROOT)
print('File count:', len(all_files))
print('Extension counts:', Counter(p.suffix.lower() for p in all_files))
print('Directories containing 003 or RedEdge:')
for path in sorted({p.parent for p in all_files if '003' in str(p) or 'rededge' in str(p).lower()}):
    print(' ', path)
print('Candidate image/label files:')
for path in all_files:
    low = str(path).lower()
    if ('003' in low and any(token in low for token in ('groundtruth', 'imap', 'rededge'))):
        print(path)

metadata_files = [p for p in all_files if p.suffix.lower() in ('.txt', '.md', '.xml', '.json', '.csv', '.yaml', '.yml')]
for path in metadata_files[:80]:
    print('METADATA', path, 'sha256=', sha256_file(path))

label_candidates = [p for p in all_files if '003' in str(p) and 'groundtruth' in p.name.lower()]
assert label_candidates, 'No GroundTruth file under dataset 003; inspect archive layout before proceeding'
label_samples = []
for path in label_candidates[:10]:
    with Image.open(path) as label_image:
        print('LABEL', path, 'mode=', label_image.mode, 'size=', label_image.size,
              'bands=', label_image.getbands(), 'format=', label_image.format)
        print('  palette/info keys:', sorted(label_image.info.keys()))
        label_samples.append({'path': str(path), 'mode': label_image.mode,
                              'size': list(label_image.size), 'bands': list(label_image.getbands()),
                              'format': label_image.format,
                              'info_keys': sorted(label_image.info.keys())})

rededge_candidates = [p for p in all_files if '003' in str(p) and 'rededge' in str(p).lower()]
assert rededge_candidates, 'Dataset 003 RedEdge files not found; do not substitute Sequoia'
print('RedEdge-M candidate count:', len(rededge_candidates))
archive_candidates = [p for p in all_files if p.suffix.lower() in ('.zip', '.tar', '.gz')
                      and ('003' in p.name or 'weedmap' in p.name.lower())]
archive_hashes = {str(path): sha256_file(path) for path in archive_candidates}
preflight = {
    'source_page': 'https://projects.asl.ethz.ch/datasets/weedmap-2018/',
    'paper_doi': '10.3390/rs10091423',
    'dataset_root': WM_ROOT,
    'file_count': len(all_files),
    'dataset_003_rededge_candidate_count': len(rededge_candidates),
    'rededge_sample_paths': [str(path) for path in rededge_candidates[:100]],
    'label_samples': label_samples,
    'metadata_files': [str(path) for path in metadata_files],
    'archive_hashes': archive_hashes,
    'manual_review_decision': 'PENDING',
    'manual_review_fields': ['official partition 003 verified from source docs',
                             'five co-registered RedEdge-M bands and order verified',
                             'radiometric scale fixed without test-fitted statistics',
                             'label mapping/license/instance availability verified',
                             'tile/orthomosaic grouping and leakage reviewed'],
    'preflight_note': 'No inference performed. Change decision to GO only after reviewing all fields.',
}
with open(f'{OUT}/weedmap_003_preflight.json', 'w') as f:
    json.dump(preflight, f, indent=2)
print('Saved:', f'{OUT}/weedmap_003_preflight.json')
print('Archive inspection only. Do not evaluate model predictions or tune from these labels.')

Mounted root: /kaggle/working/weedmap-2018
File count: 5119
Extension counts: Counter({'.png': 5100, '.zip': 10, '': 6, '.ds_store': 3})
Directories containing 003 or RedEdge:
  /kaggle/working/weedmap-2018/Tiles
  /kaggle/working/weedmap-2018/Tiles/RedEdge
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/groundtruth
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/mask
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/B
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/CIR
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/G
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/NDVI
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/NIR
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/R
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/RE
  /kaggle/working/weedmap-2018/Tiles/RedEdge/000/tile/RGB
  /kaggle/working/weedmap-2018/Tiles/RedEdge/003/003
  /kaggle/wor

In [18]:
#Cell K8.1 — Profile L0, OccCond-g, MCSD-g, PS (600×600)
import json, os, subprocess, sys

OUT = 'results/exp08'
LOGS = f'{OUT}/logs'
os.makedirs(LOGS, exist_ok=True)

def run_live(args, log_path):
    print('\n$', ' '.join(args), flush=True)
    with open(log_path, 'w', buffering=1) as log_file:
        process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
        code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, args)

profiles = {
    'L0': '--ccmc=False --head=bilinear --use_low_level_refine=False',
    'PS': '--ccmc=False --head=pixelshuffle --spectral_guide=False',
    'OccCondg': '--ccmc=False --head=occ_conditioned --spectral_guide=True --mcsd_tau=1.0',
    'MCSDg': '--ccmc=False --head=mcsd --spectral_guide=True --sinkhorn_iters=5 --mcsd_tau=1.0',
}
for label, arch in profiles.items():
    out_json = f'{OUT}/profile_{label}.json'
    args = [sys.executable, 'src/scripts/profile_model.py', *arch.split(),
            '--in_channels=5', '--num_classes=3', '--use_multiscale_fusion=True',
            '--size=600', '--warmup=50', '--runs=300', f'--label={label}', f'--out={out_json}']
    run_live(args, f'{LOGS}/profile_{label}.log')
    assert os.path.isfile(out_json), f'{label} profile JSON missing'
    with open(out_json) as f:
        result = json.load(f)
    print(label, 'params_M=', result['params_M'],
          'GFLOPs=', result['GFLOPs_flopcounter'],
          'median_ms=', result['latency_ms_median'],
          'peak_mem_MB=', result['peak_mem_MB'], flush=True)

cost_rows = []
for label in profiles:
    with open(f'{OUT}/profile_{label}.json') as f:
        result = json.load(f)
    cost_rows.append({
        'method': label,
        'params_M': result['params_M'],
        'GFLOPs': result['GFLOPs_flopcounter'],
        'GMACs': result['GMACs'],
        'latency_ms_median': result['latency_ms_median'],
        'latency_ms_p10': result['latency_ms_p10'],
        'latency_ms_p90': result['latency_ms_p90'],
        'peak_mem_MB': result['peak_mem_MB'],
        'device': result['device'],
        'input': result['input'],
        'precision': result['precision'],
    })
cost_path = f'{OUT}/compute_cost_table.json'
with open(cost_path, 'w') as f:
    json.dump({'protocol': '600x600, batch=1, fp32, warmup=50, synchronized runs=300',
               'profiles': cost_rows}, f, indent=2)
print('Saved compute-cost table:', cost_path, flush=True)
for row in cost_rows:
    print(row, flush=True)


$ /usr/bin/python3 src/scripts/profile_model.py --ccmc=False --head=bilinear --use_low_level_refine=False --in_channels=5 --num_classes=3 --use_multiscale_fusion=True --size=600 --warmup=50 --runs=300 --label=L0 --out=results/exp08/profile_L0.json
{
  "label": "L0",
  "input": [
    1,
    5,
    600,
    600
  ],
  "device": "Tesla T4",
  "torch": "2.11.0+cu128",
  "precision": "fp32",
  "batch": 1,
  "params_M": 3.455619,
  "prior_head_params": 0,
  "GFLOPs_flopcounter": 13.268002048,
  "GMACs": 6.634001024,
  "modulation_elementwise_Mops": 0.0,
  "latency_ms_median": 14.225264072418213,
  "latency_ms_p10": 14.162086009979248,
  "latency_ms_p90": 14.307817554473877,
  "latency_ms_mean": 14.234205360412597,
  "runs": 300,
  "warmup": 50,
  "peak_mem_MB": 145.3662109375
}
L0 params_M= 3.455619 GFLOPs= 13.268002048 median_ms= 14.225264072418213 peak_mem_MB= 145.3662109375

$ /usr/bin/python3 src/scripts/profile_model.py --ccmc=False --head=pixelshuffle --spectral_guide=False --in_chann

In [19]:
#cell kiểm toán chính xác partition 003
from pathlib import Path
from PIL import Image
import numpy as np
from collections import Counter

ROOT = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
TILE = ROOT / 'tile'
GT = ROOT / 'groundtruth'
MASK = ROOT / 'mask'
BANDS = ('R', 'G', 'B', 'NIR', 'RE')

assert all((TILE / b).is_dir() for b in BANDS)
assert GT.is_dir() and MASK.is_dir()

sets = {
    b: {p.stem for p in (TILE / b).glob('frame*.png')}
    for b in BANDS
}
common = set.intersection(*sets.values())

print('Frames per band:', {b: len(v) for b, v in sets.items()})
print('Complete five-band frames:', len(common))
for b in BANDS:
    print('Missing/alignment IDs:', b, len(sets[b] - common))

for frame in sorted(common)[:5]:
    print('\nFRAME', frame)
    shapes = []
    for b in BANDS:
        with Image.open(TILE / b / f'{frame}.png') as im:
            x = np.asarray(im)
            shapes.append(x.shape)
            print(b, 'mode:', im.mode, 'dtype:', x.dtype,
                  'shape:', x.shape, 'range:', (x.min(), x.max()))
    assert len(set(shapes)) == 1, 'Band shape mismatch'
    assert all(len(s) == 2 for s in shapes), 'Expected single-channel bands'

    prefix = f'003_{frame}'
    gt_files = sorted(GT.glob(prefix + '*.png'))
    print('GT files:', [p.name for p in gt_files])

    for p in gt_files:
        with Image.open(p) as im:
            x = np.asarray(im)
            print(' ', p.name, 'mode:', im.mode,
                  'shape:', x.shape, 'dtype:', x.dtype)

    mask_path = MASK / f'{frame}.png'
    print('Mask exists:', mask_path.exists())

print('\nExact partition audit complete.')


Frames per band: {'R': 204, 'G': 204, 'B': 204, 'NIR': 204, 'RE': 204}
Complete five-band frames: 204
Missing/alignment IDs: R 0
Missing/alignment IDs: G 0
Missing/alignment IDs: B 0
Missing/alignment IDs: NIR 0
Missing/alignment IDs: RE 0

FRAME frame0000
R mode: L dtype: uint8 shape: (360, 480) range: (np.uint8(0), np.uint8(0))
G mode: L dtype: uint8 shape: (360, 480) range: (np.uint8(0), np.uint8(0))
B mode: L dtype: uint8 shape: (360, 480) range: (np.uint8(0), np.uint8(0))
NIR mode: L dtype: uint8 shape: (360, 480) range: (np.uint8(0), np.uint8(0))
RE mode: L dtype: uint8 shape: (360, 480) range: (np.uint8(0), np.uint8(0))
GT files: ['003_frame0000.png', '003_frame0000_GroundTruth_color.png', '003_frame0000_GroundTruth_iMap.png']
  003_frame0000.png mode: L shape: (360, 480) dtype: uint8
  003_frame0000_GroundTruth_color.png mode: L shape: (360, 480) dtype: uint8
  003_frame0000_GroundTruth_iMap.png mode: L shape: (360, 480) dtype: uint8
Mask exists: True

FRAME frame0001
R mode: L

In [20]:
#kiểm kê toàn bộ 204 frame mà không chạy mô hình
from pathlib import Path
from PIL import Image
import numpy as np
from collections import Counter

ROOT = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
BANDS = ('R', 'G', 'B', 'NIR', 'RE')

rows = []
frames = sorted(p.stem for p in (ROOT/'tile'/'R').glob('frame*.png'))

for frame in frames:
    xs = {}
    for band in BANDS:
        with Image.open(ROOT/'tile'/band/f'{frame}.png') as im:
            xs[band] = np.asarray(im)

    with Image.open(ROOT/'mask'/f'{frame}.png') as im:
        mask = np.asarray(im)

    active = np.stack([xs[b] != 0 for b in BANDS], axis=0)
    rows.append({
        'frame': frame,
        'all_bands_zero': not active.any(),
        'nonzero_any_fraction': float(active.any(axis=0).mean()),
        'nonzero_all_fraction': float(active.all(axis=0).mean()),
        'mask_values': tuple(int(v) for v in np.unique(mask)),
        'mask_nonzero_fraction': float(np.mean(mask != 0)),
        'band_max': {b: int(xs[b].max()) for b in BANDS},
    })

print('Total frames:', len(rows))
print('All-band zero:', sum(r['all_bands_zero'] for r in rows))
print('At least one nonzero:', sum(not r['all_bands_zero'] for r in rows))
print('Mask patterns:', Counter(r['mask_values'] for r in rows))
print('Mask nonzero range:',
      min(r['mask_nonzero_fraction'] for r in rows),
      max(r['mask_nonzero_fraction'] for r in rows))

print('\nFirst 8 nonzero frames:')
for r in [r for r in rows if not r['all_bands_zero']][:8]:
    print(r)

print('\nFirst 8 all-zero frames:')
for r in [r for r in rows if r['all_bands_zero']][:8]:
    print(r['frame'], r['mask_values'], r['mask_nonzero_fraction'])

Total frames: 204
All-band zero: 102
At least one nonzero: 102
Mask patterns: Counter({(255,): 102, (0, 255): 57, (0,): 45})
Mask nonzero range: 0.0 1.0

First 8 nonzero frames:
{'frame': 'frame0007', 'all_bands_zero': False, 'nonzero_any_fraction': 0.3592708333333333, 'nonzero_all_fraction': 0.3592708333333333, 'mask_values': (0, 255), 'mask_nonzero_fraction': 0.6407291666666667, 'band_max': {'R': 189, 'G': 213, 'B': 169, 'NIR': 182, 'RE': 210}}
{'frame': 'frame0008', 'all_bands_zero': False, 'nonzero_any_fraction': 0.81, 'nonzero_all_fraction': 0.81, 'mask_values': (0, 255), 'mask_nonzero_fraction': 0.19, 'band_max': {'R': 199, 'G': 225, 'B': 197, 'NIR': 252, 'RE': 229}}
{'frame': 'frame0009', 'all_bands_zero': False, 'nonzero_any_fraction': 0.4429976851851852, 'nonzero_all_fraction': 0.4429976851851852, 'mask_values': (0, 255), 'mask_nonzero_fraction': 0.5570023148148148, 'band_max': {'R': 209, 'G': 233, 'B': 222, 'NIR': 222, 'RE': 211}}
{'frame': 'frame0010', 'all_bands_zero': Fals

In [21]:
#xác minh mask và thống kê tile hợp lệ
from pathlib import Path
from PIL import Image
import numpy as np

ROOT = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
BANDS = ('R', 'G', 'B', 'NIR', 'RE')

stats = []
for path in sorted((ROOT/'tile'/'R').glob('frame*.png')):
    frame = path.stem
    bands = np.stack([
        np.asarray(Image.open(ROOT/'tile'/b/f'{frame}.png'))
        for b in BANDS
    ])
    mask = np.asarray(Image.open(ROOT/'mask'/f'{frame}.png'))

    signal = np.any(bands != 0, axis=0)
    valid = mask == 0

    stats.append({
        'frame': frame,
        'valid_pixels': int(valid.sum()),
        'signal_pixels': int(signal.sum()),
        'mismatch_pixels': int(np.count_nonzero(signal != valid)),
    })

print('Total:', len(stats))
print('Mask nonempty:', sum(s['valid_pixels'] > 0 for s in stats))
print('Mask all invalid:', sum(s['valid_pixels'] == 0 for s in stats))
print('Mask fully valid:', sum(s['valid_pixels'] == 360*480 for s in stats))
print('Total mismatched pixels:',
      sum(s['mismatch_pixels'] for s in stats))
print('Frames with mismatches:',
      sum(s['mismatch_pixels'] > 0 for s in stats))
print('Top mismatches:',
      sorted(stats, key=lambda s: -s['mismatch_pixels'])[:5])

Total: 204
Mask nonempty: 102
Mask all invalid: 102
Mask fully valid: 45
Total mismatched pixels: 0
Frames with mismatches: 0
Top mismatches: [{'frame': 'frame0000', 'valid_pixels': 0, 'signal_pixels': 0, 'mismatch_pixels': 0}, {'frame': 'frame0001', 'valid_pixels': 0, 'signal_pixels': 0, 'mismatch_pixels': 0}, {'frame': 'frame0002', 'valid_pixels': 0, 'signal_pixels': 0, 'mismatch_pixels': 0}, {'frame': 'frame0003', 'valid_pixels': 0, 'signal_pixels': 0, 'mismatch_pixels': 0}, {'frame': 'frame0004', 'valid_pixels': 0, 'signal_pixels': 0, 'mismatch_pixels': 0}]


In [22]:
#xác minh ground truth và radiometry
#Một chỉnh sửa nhỏ: Counter ở cell này đếm số frame xuất hiện từng mã nhãn, 
#không phải số pixel của từng lớp. Vì mục tiêu chỉ là xác minh các mã xuất hiện, như vậy là đủ.
from pathlib import Path
from PIL import Image
import numpy as np
from collections import Counter

root = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
frames = sorted(p.stem for p in (root/'tile'/'R').glob('frame*.png'))

label_values = Counter()
imap_values = Counter()
gt_shape_errors = []
zero_outside_mask = 0
total_invalid = 0

for frame in frames:
    mask = np.asarray(Image.open(root/'mask'/f'{frame}.png'))
    valid = mask == 0
    prefix = f'003_{frame}'

    semantic_path = root/'groundtruth'/f'{prefix}.png'
    imap_path = root/'groundtruth'/f'{prefix}_GroundTruth_iMap.png'

    for path in (semantic_path, imap_path):
        assert path.is_file(), f'Missing: {path}'

    sem = np.asarray(Image.open(semantic_path))
    imap = np.asarray(Image.open(imap_path))

    if sem.shape != mask.shape or imap.shape != mask.shape:
        gt_shape_errors.append(frame)

    # Inventory only; do not change class mapping from these frequencies
    label_values.update(np.unique(sem[valid]).tolist())
    imap_values.update(np.unique(imap[valid]).tolist())

    for band in ('R', 'G', 'B', 'NIR', 'RE'):
        arr = np.asarray(Image.open(root/'tile'/band/f'{frame}.png'))
        zero_outside_mask += int(np.count_nonzero(arr[~valid] == 0))
        total_invalid += int((~valid).sum())

print('Semantic label codes (valid pixels):', sorted(label_values))
print('iMap codes (valid pixels):', sorted(imap_values))
print('GT shape errors:', gt_shape_errors)
print('Invalid spectral pixels equal zero:',
      zero_outside_mask, '/', total_invalid)


Semantic label codes (valid pixels): [0, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 2

In [23]:
#kiểm tra chính xác shape, nhãn iMap và mask

from pathlib import Path
from PIL import Image
import numpy as np
from collections import Counter

ROOT = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
frames = sorted(p.stem for p in (ROOT/'tile'/'R').glob('frame*.png'))

shape_patterns = Counter()
code_by_mask = {0: Counter(), 255: Counter()}
shape_mismatch = []
matched = 0

for frame in frames:
    prefix = f'003_{frame}'
    paths = {
        'R': ROOT/'tile'/'R'/f'{frame}.png',
        'mask': ROOT/'mask'/f'{frame}.png',
        'original': ROOT/'groundtruth'/f'{prefix}.png',
        'color': ROOT/'groundtruth'/f'{prefix}_GroundTruth_color.png',
        'imap': ROOT/'groundtruth'/f'{prefix}_GroundTruth_iMap.png',
    }
    imgs = {}
    modes = {}

    for name, path in paths.items():
        with Image.open(path) as im:
            imgs[name] = np.asarray(im)
            modes[name] = im.mode

    signature = tuple(
        (name, imgs[name].shape, modes[name])
        for name in ('R', 'mask', 'original', 'color', 'imap')
    )
    shape_patterns[signature] += 1

    rshape = imgs['R'].shape
    if (imgs['mask'].shape != rshape or
        imgs['imap'].shape != rshape):
        shape_mismatch.append({
            'frame': frame,
            'R': rshape,
            'mask': imgs['mask'].shape,
            'imap': imgs['imap'].shape,
            'mode_imap': modes['imap']
        })
        continue

    matched += 1
    imap = imgs['imap']
    mask = imgs['mask']

    for mv in (0, 255):
        values, counts = np.unique(
            imap[mask == mv], return_counts=True
        )
        for value, count in zip(values, counts):
            code_by_mask[mv][int(value)] += int(count)

print('Total frames:', len(frames))
print('Shape patterns:')
for signature, count in shape_patterns.most_common(8):
    print('Count:', count)
    for name, shape, mode in signature:
        print(' ', name, shape, mode)

print('\niMap-mask aligned frames:', matched)
print('iMap shape mismatches:', len(shape_mismatch))
print('First mismatches:', shape_mismatch[:10])

for mask_value in (0, 255):
    print(f'\niMap pixel counts where mask={mask_value}:')
    print(code_by_mask[mask_value].most_common(20))

print('\nAudit completed. No resizing or relabeling.')


Total frames: 204
Shape patterns:
Count: 111
  R (360, 480) L
  mask (360, 480) L
  original (360, 480) L
  color (360, 480) L
  imap (360, 480) L
Count: 93
  R (360, 480) L
  mask (360, 480) L
  original (360, 480, 3) RGB
  color (360, 480, 3) RGB
  imap (360, 480) I;16

iMap-mask aligned frames: 204
iMap shape mismatches: 0
First mismatches: []

iMap pixel counts where mask=0:
[(0, 12000413), (10000, 668227), (2, 343799)]

iMap pixel counts where mask=255:
[(0, 22238632), (2, 127), (10000, 2)]

Audit completed. No resizing or relabeling.


In [24]:
#Chạy cell dưới đây. Nó chỉ làm data-integrity audit, không suy luận bằng model.

from pathlib import Path
from PIL import Image
import numpy as np
from collections import Counter, defaultdict

root = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
frames = sorted(p.stem for p in (root/'tile'/'R').glob('frame*.png'))

groups = defaultdict(list)
label_codes = defaultdict(Counter)

for frame in frames:
    prefix = f'003_{frame}'
    with Image.open(root/'tile'/'R'/f'{frame}.png') as im:
        r = np.asarray(im)
    with Image.open(root/'mask'/f'{frame}.png') as im:
        mask = np.asarray(im)
    with Image.open(root/'groundtruth'/f'{prefix}_GroundTruth_iMap.png') as im:
        mode = im.mode
        gt = np.asarray(im)

    has_signal = bool(np.any(r != 0))
    key = (has_signal, mode)
    groups[key].append(frame)

    valid = mask == 0
    values, counts = np.unique(gt[valid], return_counts=True)
    for value, count in zip(values, counts):
        label_codes[key][int(value)] += int(count)

print('Cross-tabulation:')
for key, ids in sorted(groups.items()):
    print('\nSignal, iMap mode:', key)
    print('Number of frames:', len(ids))
    print('Label codes/counts:', label_codes[key])
    print('First frame IDs:', ids[:15])

print('\nNonzero spectral frames with non-I;16 labels:')
for (has_signal, mode), ids in groups.items():
    if has_signal and mode != 'I;16':
        print(mode, ids)

print('\nCrop class pixel count:',
      sum(c.get(1, 0) for c in label_codes.values()))


Cross-tabulation:

Signal, iMap mode: (False, 'L')
Number of frames: 102
Label codes/counts: Counter()
First frame IDs: ['frame0000', 'frame0001', 'frame0002', 'frame0003', 'frame0004', 'frame0005', 'frame0006', 'frame0011', 'frame0012', 'frame0013', 'frame0014', 'frame0015', 'frame0016', 'frame0017', 'frame0023']

Signal, iMap mode: (True, 'I;16')
Number of frames: 93
Label codes/counts: Counter({0: 11869728, 10000: 668227, 2: 343799})
First frame IDs: ['frame0007', 'frame0008', 'frame0009', 'frame0018', 'frame0019', 'frame0020', 'frame0021', 'frame0022', 'frame0029', 'frame0030', 'frame0031', 'frame0032', 'frame0033', 'frame0034', 'frame0041']

Signal, iMap mode: (True, 'L')
Number of frames: 9
Label codes/counts: Counter({0: 130685})
First frame IDs: ['frame0010', 'frame0035', 'frame0040', 'frame0075', 'frame0138', 'frame0172', 'frame0180', 'frame0193', 'frame0194']

Nonzero spectral frames with non-I;16 labels:
L ['frame0010', 'frame0035', 'frame0040', 'frame0075', 'frame0138', 'fr

In [25]:
#Chạy cell sau trên một số frame I;16. 
#Nó chỉ xem giá trị pixel tại những vị trí tương ứng giữa ảnh nhãn màu và iMap; không dùng kết quả để chọn mô hình hay đánh giá hiệu năng.
from pathlib import Path
from PIL import Image
import numpy as np

root = Path('/kaggle/working/weedmap-2018/Tiles/RedEdge/003/003')
frames = ['frame0007', 'frame0019', 'frame0020', 'frame0030']

for frame in frames:
    prefix = f'003_{frame}'
    gt_dir = root / 'groundtruth'

    with Image.open(gt_dir/f'{prefix}_GroundTruth_iMap.png') as im:
        ids = np.asarray(im)
    with Image.open(gt_dir/f'{prefix}_GroundTruth_color.png') as im:
        rgb = np.asarray(im.convert('RGB'))
    with Image.open(root/'mask'/f'{frame}.png') as im:
        valid = np.asarray(im) == 0

    print('\n', frame)
    for code in (0, 1, 2, 10000):
        selected = (ids == code) & valid
        n = int(selected.sum())
        if n == 0:
            print('code', code, ': absent')
            continue

        colors, counts = np.unique(
            rgb[selected].reshape(-1, 3),
            axis=0,
            return_counts=True
        )
        top = np.argsort(counts)[-5:][::-1]
        print('code', code, 'pixels', n,
              'color mappings:',
              [(colors[i].tolist(), int(counts[i])) for i in top])


 frame0007
code 0 pixels 59402 color mappings: [([0, 0, 0], 59402)]
code 1 : absent
code 2 pixels 2680 color mappings: [([255, 0, 0], 2680)]
code 10000 : absent

 frame0019
code 0 pixels 160626 color mappings: [([0, 0, 0], 160626)]
code 1 : absent
code 2 pixels 2926 color mappings: [([255, 0, 0], 2926)]
code 10000 pixels 8961 color mappings: [([0, 255, 0], 8961)]

 frame0020
code 0 pixels 155723 color mappings: [([0, 0, 0], 155723)]
code 1 : absent
code 2 pixels 5081 color mappings: [([255, 0, 0], 5081)]
code 10000 pixels 11996 color mappings: [([0, 255, 0], 11996)]

 frame0030
code 0 pixels 164995 color mappings: [([0, 0, 0], 164995)]
code 1 : absent
code 2 pixels 912 color mappings: [([255, 0, 0], 912)]
code 10000 pixels 4260 color mappings: [([0, 255, 0], 4260)]


In [28]:
from pathlib import Path

ROOT = Path('/kaggle/working/weedmap-2018/Tiles')

for p in sorted(ROOT.iterdir()):
    print(('DIR  ' if p.is_dir() else 'FILE '), p.name)

print('\nZIP files:')
for p in ROOT.rglob('*.zip'):
    print(p, f'{p.stat().st_size / 1024 / 1024:.2f} MB')

DIR   RedEdge
FILE  RedEdge.zip
DIR   Sequoia
FILE  Sequoia.zip

ZIP files:
/kaggle/working/weedmap-2018/Tiles/Sequoia.zip 423.57 MB
/kaggle/working/weedmap-2018/Tiles/RedEdge.zip 608.35 MB
/kaggle/working/weedmap-2018/Tiles/Sequoia/005.zip 179.10 MB
/kaggle/working/weedmap-2018/Tiles/Sequoia/007.zip 115.17 MB
/kaggle/working/weedmap-2018/Tiles/Sequoia/006.zip 129.18 MB
/kaggle/working/weedmap-2018/Tiles/RedEdge/002.zip 186.65 MB
/kaggle/working/weedmap-2018/Tiles/RedEdge/004.zip 80.94 MB
/kaggle/working/weedmap-2018/Tiles/RedEdge/001.zip 106.50 MB
/kaggle/working/weedmap-2018/Tiles/RedEdge/003.zip 106.32 MB
/kaggle/working/weedmap-2018/Tiles/RedEdge/000.zip 127.76 MB


In [26]:
#Cell K8.0b — Read WeedMap annotation documentation/source only (no label pixels)
import hashlib
import os
import pathlib
import re

WM_ROOT = pathlib.Path('/kaggle/working/weedmap-2018')
AUDIT_LOG = pathlib.Path('results/exp08/logs/weedmap_annotation_docs_audit.log')
AUDIT_LOG.parent.mkdir(parents=True, exist_ok=True)
assert WM_ROOT.is_dir(), f'WeedMap root not found: {WM_ROOT}'

text_extensions = {
    '.txt', '.md', '.rst', '.html', '.htm', '.xml', '.json', '.yaml', '.yml',
    '.csv', '.m', '.py', '.tex', '.ini', '.cfg'
}
search_pattern = re.compile(
    r'10000|ground.?truth.?imap|ground.?truth.?color|background|crop|weed|'
    r'ignore|unlabel|non.?class|annotation|label.?map|class.?code',
    re.IGNORECASE,
)
exact_code_pattern = re.compile(r'(?<!\d)10000(?!\d)|GroundTruth[_ -]?(?:iMap|color)', re.IGNORECASE)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def emit(message=''):
    print(message, flush=True)
    log.write(message + '\n')

text_files = sorted(
    path for path in WM_ROOT.rglob('*')
    if path.is_file() and path.suffix.lower() in text_extensions
    and path.stat().st_size <= 4 * 1024 * 1024
)
exact_hits = 0
files_with_hits = 0
with AUDIT_LOG.open('w', encoding='utf-8', buffering=1) as log:
    emit('WeedMap annotation documentation/source audit')
    emit(f'Root: {WM_ROOT}')
    emit('Scope: text files only; no PNG/TIFF/mask pixels opened.')
    emit(f'Text files scanned: {len(text_files)} (max 4 MiB each)')
    for path in text_files:
        try:
            lines = path.read_text(encoding='utf-8-sig', errors='replace').splitlines()
        except OSError as error:
            emit(f'SKIP {path.relative_to(WM_ROOT)}: {error}')
            continue
        matches = [(index, line.strip()) for index, line in enumerate(lines, start=1)
                   if search_pattern.search(line)]
        exact_matches = [(index, line) for index, line in matches
                         if exact_code_pattern.search(line)]
        if not matches:
            continue
        files_with_hits += 1
        exact_hits += len(exact_matches)
        relative = path.relative_to(WM_ROOT)
        emit(f'\nFILE {relative} sha256={sha256_file(path)}')
        selected = exact_matches if exact_matches else matches[:5]
        if exact_matches:
            context_lines = set()
            for index, _ in exact_matches:
                context_lines.update(range(max(1, index - 2), min(len(lines), index + 2) + 1))
            selected = [(index, lines[index - 1].strip()) for index in sorted(context_lines)]
        else:
            selected = matches[:5]
        for index, line in selected[:20]:
            emit(f'  L{index}: {line[:500]}')
    emit('\nAUDIT RESULT: PENDING')
    emit('Only mark semantics verified if an official release document or author source explicitly defines code 10000 and the class/ignore policy.')
    emit(f'Files with relevant text: {files_with_hits}; exact code/name hits: {exact_hits}')
    emit(f'Saved audit: {AUDIT_LOG}')
print('Saved audit:', AUDIT_LOG, flush=True)

WeedMap annotation documentation/source audit
Root: /kaggle/working/weedmap-2018
Scope: text files only; no PNG/TIFF/mask pixels opened.
Text files scanned: 0 (max 4 MiB each)

AUDIT RESULT: PENDING
Only mark semantics verified if an official release document or author source explicitly defines code 10000 and the class/ignore policy.
Files with relevant text: 0; exact code/name hits: 0
Saved audit: results/exp08/logs/weedmap_annotation_docs_audit.log
Saved audit: results/exp08/logs/weedmap_annotation_docs_audit.log


In [14]:

# ============================================================
# Cell K7.0 — Setup, test và checkpoint check
# ============================================================

import glob
import json
import os
import shutil
import subprocess
import sys

# ------------------------------------------------------------
# 1. Project setup
# ------------------------------------------------------------

PROJECT = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(PROJECT)

if 'DS' not in globals():
    candidates = glob.glob(
        '/kaggle/input/**/weedsgalore-dataset',
        recursive=True
    )
    assert candidates, (
        'Cannot find weedsgalore-dataset under /kaggle/input'
    )
    DS = candidates[0]

assert os.path.isdir(DS), f'Invalid dataset path: {DS}'

EXP = 'exp07'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'

os.makedirs(f'{OUT}/logs', exist_ok=True)


# ------------------------------------------------------------
# 2. Helper function
# ------------------------------------------------------------

def run_checked(args, log_path):
    with open(log_path, 'w') as log_file:
        subprocess.run(
            args,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=True
        )


# ------------------------------------------------------------
# 3. Unit tests
# ------------------------------------------------------------

print('=' * 60)
print('STEP 1: Running unit and integration tests')
print('=' * 60)

run_checked(
    [
        sys.executable,
        '-m',
        'pytest',
        '-q',
        'tests/test_ccmc_sr.py'
    ],
    f'{OUT}/logs/unit_tests.log'
)

with open(f'{OUT}/logs/unit_tests.log') as f:
    test_output = f.read().lower()

print(test_output)

assert ' skipped' not in test_output, (
    'HF-backed integration tests were skipped; '
    'do not start training'
)

print('PASS: Unit and integration tests')


# ------------------------------------------------------------
# 4. Link Exp06 checkpoints
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('STEP 2: Checking Exp06 checkpoints')
print('=' * 60)

checkpoint_candidates = glob.glob(
    '/kaggle/input/**/runs/*/best_val.pth',
    recursive=True
)

for path in checkpoint_candidates:
    run_dir = os.path.basename(os.path.dirname(path))
    target_dir = os.path.join('runs', run_dir)

    os.makedirs(target_dir, exist_ok=True)

    target = os.path.join(target_dir, 'best_val.pth')

    if not os.path.exists(target):
        os.symlink(path, target)

required = [
    f'{model}_s{seed}'
    for model in ('L0', 'MCSDg', 'PS')
    for seed in range(5)
]

missing = [
    run for run in required
    if not os.path.isfile(f'runs/{run}/best_val.pth')
]

assert not missing, (
    f'Missing Exp06 reference checkpoints: {missing}'
)

print('PASS: All 15 Exp06 checkpoints verified')


# ------------------------------------------------------------
# 5. Exp06 validation results
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('STEP 3: Checking Exp06 validation results')
print('=' * 60)

# Actual existing directory in Kaggle working
EXP06_SOURCE = (
    '/kaggle/working/WeedsGalore_Ipr28/'
    'doc/exp06_results/results/exp06'
)

assert os.path.isdir(EXP06_SOURCE), (
    f'Exp06 results directory not found: {EXP06_SOURCE}'
)

required_results = [
    f'{model}_s{seed}_val_{suffix}'
    for model in (
        'L0',
        'PS',
        'MCSDg',
        'MCSDgIt1',
        'MCSDgIt20'
    )
    for seed in range(5)
    for suffix in (
        'per_image.csv',
        'summary.json'
    )
]

missing_results = [
    name for name in required_results
    if not os.path.isfile(
        os.path.join(EXP06_SOURCE, name)
    )
]

assert not missing_results, (
    f'Missing Exp06 validation files: {missing_results}'
)

print('PASS: All 50 source validation files found')


# ------------------------------------------------------------
# 6. Copy Exp06 reference results into Exp07
# ------------------------------------------------------------

EXP06_RESULTS = f'{OUT}/exp06_reference'

os.makedirs(EXP06_RESULTS, exist_ok=True)

for filename in required_results:
    shutil.copy2(
        os.path.join(EXP06_SOURCE, filename),
        os.path.join(EXP06_RESULTS, filename)
    )

copied_files = [
    f for f in required_results
    if os.path.isfile(
        os.path.join(EXP06_RESULTS, f)
    )
]

assert len(copied_files) == 50, (
    f'Expected 50 copied files, got {len(copied_files)}'
)

print('PASS: Exp06 reference files copied:', len(copied_files))
print('Reference directory:', EXP06_RESULTS)


# ------------------------------------------------------------
# 7. Generate instance bins
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('STEP 4: Generating instance bins')
print('=' * 60)

run_checked(
    [
        sys.executable,
        'src/scripts/make_instance_bins.py',
        f'--dataset_path={DS}',
        f'--out={BINS}'
    ],
    f'{OUT}/logs/bins.log'
)

assert os.path.isfile(BINS), (
    f'Instance bins file not generated: {BINS}'
)

with open(BINS) as f:
    bins = json.load(f)

assert bins['edges'] == [45.0, 147.0], (
    f'Unexpected instance bin edges: {bins["edges"]}'
)

print('PASS: Instance bins verified:', bins['edges'])


# ------------------------------------------------------------
# 8. Final checkpoint
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('CELL K7.0 — ALL CHECKS PASSED')
print('=' * 60)

print('Project:', PROJECT)
print('Dataset:', DS)
print('Experiment:', EXP)
print('Output:', OUT)
print('Checkpoints verified:', len(required))
print('Exp06 validation files:', len(copied_files))
print('Instance bin edges:', bins['edges'])
print('Unit test log:', f'{OUT}/logs/unit_tests.log')
print('Bins log:', f'{OUT}/logs/bins.log')

print('\nREADY FOR NEXT EXP07 SETUP STEP')


STEP 1: Running unit and integration tests
......................................................                   [100%]
54 passed in 36.99s

PASS: Unit and integration tests

STEP 2: Checking Exp06 checkpoints
PASS: All 15 Exp06 checkpoints verified

STEP 3: Checking Exp06 validation results
PASS: All 50 source validation files found
PASS: Exp06 reference files copied: 50
Reference directory: results/exp07/exp06_reference

STEP 4: Generating instance bins
PASS: Instance bins verified: [45.0, 147.0]

CELL K7.0 — ALL CHECKS PASSED
Project: /kaggle/working/WeedsGalore_Ipr28
Dataset: /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset
Experiment: exp07
Output: results/exp07
Checkpoints verified: 15
Exp06 validation files: 50
Instance bin edges: [45.0, 147.0]
Unit test log: results/exp07/logs/unit_tests.log
Bins log: results/exp07/logs/bins.log

READY FOR NEXT EXP07 SETUP STEP


In [15]:
#Cell K7.1 — Train OccCond-g, seed 0–4
COMMON = (f'--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 '
		  '--batch_size=8 --num_workers=4 --lr=0.0003 '
		  '--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 '
		  '--use_dice_loss=True --dice_weight=0.3 --epochs=100 --log_interval=50 '
		  '--ckpt_interval=1000000 --pretrained_backbone=True --spectral_guided=False '
		  '--use_multiscale_fusion=True --resume=True')
ARCH = ('--ccmc=False --segformer=True --use_low_level_refine=False '
		'--head=occ_conditioned --spectral_guide=True --mcsd_tau=1.0 --occ_weight=0.3')

for seed in range(5):
	run = f'OccCondg_s{seed}'
	if not os.path.exists(f'runs/{run}/run_summary.json'):
		args = [sys.executable, 'src/train.py', *COMMON.split(), *ARCH.split(),
		        f'--seed={seed}', f'--run_name=OccCondg__s{seed}', f'--out_dir=runs/{run}']
		run_checked(args, f'{OUT}/logs/train_{run}.log')
	assert os.path.isfile(f'runs/{run}/run_summary.json'), f'{run}: missing run_summary.json'
	assert os.path.isfile(f'runs/{run}/best_val.pth'), f'{run}: missing best_val.pth'
	with open(f'runs/{run}/run_summary.json') as f:
		summary = json.load(f)
	assert summary['best_epoch'] >= 0 and summary['best_val_miou'] > 0, f'{run}: invalid summary'
	with open(f'runs/{run}/history.jsonl') as f:
		history = [json.loads(line) for line in f if line.strip()]
	assert len(history) == 100 and history[-1]['epoch'] == 99, f'{run}: incomplete training history'

In [16]:
#Cell K7.2 — Evaluate, compare
EVAL_ARCH = '--ccmc=False --head=occ_conditioned --spectral_guide=True'

def evaluate(model, seed, arch, tag=None, extra=''):
	ckpt = f'runs/{model}_s{seed}/best_val.pth'
	if not os.path.isfile(ckpt):
		raise FileNotFoundError(ckpt)
	name = f'{tag or model}_s{seed}'
	args = [sys.executable, 'src/evaluate_ccmc.py', f'--dataset_path={DS}', '--split=val',
	        f'--ckpt={ckpt}', *arch.split(), *extra.split(), f'--bins_json={BINS}',
	        '--boundary_px=3', f'--run_name={name}', f'--out_prefix={OUT}/{name}_val']
	run_checked(args, f'{OUT}/logs/eval_{name}.log')
	assert os.path.isfile(f'{OUT}/{name}_val_per_image.csv'), f'{name}: missing per-image CSV'
	assert os.path.isfile(f'{OUT}/{name}_val_summary.json'), f'{name}: missing summary JSON'

for seed in range(5):
	evaluate('OccCondg', seed, EVAL_ARCH)

def evaluation_csv(model, seed):
	root = OUT if model == 'OccCondg' else EXP06_RESULTS
	path = os.path.join(root, f'{model}_s{seed}_val_per_image.csv')
	if not os.path.isfile(path):
		raise FileNotFoundError(path)
	return path

def compare(a, b):
	common = list(range(5))
	files_a = ','.join(evaluation_csv(a, seed) for seed in common)
	files_b = ','.join(evaluation_csv(b, seed) for seed in common)
	out_path = f'{OUT}/cmp_{a}_vs_{b}.json'
	args = [sys.executable, 'src/scripts/compare_runs.py', f'--a={files_a}', f'--b={files_b}',
	        f'--name_a={a}', f'--name_b={b}', '--gate=none', '--seed_bootstrap=True',
	        '--metrics=SmallWeedRecall,recall30_small,cov_small,mIoU,IoU_crop,IoU_weed,BIoU_weed,PrecisionWeed,CropToWeedRate',
	        f'--out={out_path}']
	run_checked(args, f'{OUT}/logs/cmp_{a}_vs_{b}.log')
	assert os.path.isfile(out_path), f'{a} vs {b}: comparison JSON missing'
	with open(out_path) as f:
		assert 'PrecisionWeed' in json.load(f)['metrics']

for a, b in (('MCSDg', 'OccCondg'), ('MCSDg', 'L0'), ('OccCondg', 'L0'),
			 ('MCSDg', 'PS'), ('MCSDgIt1', 'MCSDg'), ('MCSDgIt20', 'MCSDg')):
	compare(a, b)

In [17]:
#Cell K7.3 — Audit marginal theo lớp/occupancy
audit_specs = [('MCSDg', 'mcsd', (1, 5, 20)), ('OccCondg', 'occ_conditioned', (5,))]
for model, head, iteration_counts in audit_specs:
	for seed in range(5):
		ckpt = f'runs/{model}_s{seed}/best_val.pth'
		for iters in iteration_counts:
			prefix = f'{OUT}/audit_{model}_s{seed}_it{iters}'
			args = [sys.executable, 'src/scripts/mcsd_audit.py', f'--dataset_path={DS}',
			        f'--ckpt={ckpt}', f'--head={head}', '--spectral_guide=True',
			        f'--sinkhorn_iters={iters}', '--mcsd_tau=1.0', '--bootstrap_samples=2000',
			        f'--out_prefix={prefix}']
			run_checked(args, f'{OUT}/logs/audit_{model}_s{seed}_it{iters}.log')
			for suffix in ('_marginal_by_image.csv', '_predweed_by_image.csv', '_marginal_summary.json'):
				assert os.path.isfile(prefix + suffix), f'Missing audit output: {prefix + suffix}'

In [18]:
#Cell K7.4 — Delta per-image cho 1 vòng so với 5 vòng
for seed in range(5):
	ckpt = f'runs/MCSDg_s{seed}/best_val.pth'
	prefix = f'{OUT}/iteration_audit_MCSDg_s{seed}_it1_vs_it5'
	args = [sys.executable, 'src/scripts/mcsd_iteration_audit.py',
	        f'--dataset_path={DS}', f'--ckpt={ckpt}', f'--bins_json={BINS}',
	        '--head=mcsd', '--spectral_guide=True', '--mcsd_tau=1.0',
	        '--iters_a=1', '--iters_b=5', '--bootstrap_samples=2000',
	        f'--out_prefix={prefix}']
	run_checked(args, f'{OUT}/logs/iteration_audit_MCSDg_s{seed}.log')
	for suffix in ('_per_image.csv', '_small_instances.csv', '_summary.json'):
		assert os.path.isfile(prefix + suffix), f'Missing iteration audit output: {prefix + suffix}'

In [24]:
import json
import os

for seed in range(5):
    path = f'runs/OccCondg_s{seed}/run_summary.json'

    with open(path) as f:
        summary = json.load(f)

    print(f'\n===== OccCondg_s{seed} =====')
    print('Top-level keys:', list(summary.keys()))

    if 'metrics' in summary:
        print('Metrics:', summary['metrics'])
    else:
        print('No top-level metrics key')

    print('Summary preview:')
    print(json.dumps(summary, indent=2)[:1500])


===== OccCondg_s0 =====
Top-level keys: ['run_name', 'best_val_miou', 'best_epoch', 'final_lambdas', 'split_hash']
No top-level metrics key
Summary preview:
{
  "run_name": "OccCondg__s0",
  "best_val_miou": 83.43894958496094,
  "best_epoch": 90,
  "final_lambdas": [],
  "split_hash": "675667edf3f35f16"
}

===== OccCondg_s1 =====
Top-level keys: ['run_name', 'best_val_miou', 'best_epoch', 'final_lambdas', 'split_hash']
No top-level metrics key
Summary preview:
{
  "run_name": "OccCondg__s1",
  "best_val_miou": 83.59940338134766,
  "best_epoch": 87,
  "final_lambdas": [],
  "split_hash": "675667edf3f35f16"
}

===== OccCondg_s2 =====
Top-level keys: ['run_name', 'best_val_miou', 'best_epoch', 'final_lambdas', 'split_hash']
No top-level metrics key
Summary preview:
{
  "run_name": "OccCondg__s2",
  "best_val_miou": 83.64779663085938,
  "best_epoch": 96,
  "final_lambdas": [],
  "split_hash": "675667edf3f35f16"
}

===== OccCondg_s3 =====
Top-level keys: ['run_name', 'best_val_miou', 'best

In [21]:
import os
import json

for seed in range(5):
    run = f'OccCondg_s{seed}'
    run_dir = f'runs/{run}'

    print(f'\n===== {run} =====')

    for filename in [
        'best_val.pth',
        'run_summary.json',
        'history.jsonl',
        'run_config.json'
    ]:
        path = os.path.join(run_dir, filename)
        print(filename, 'OK' if os.path.isfile(path) else 'MISSING')

    summary_path = os.path.join(run_dir, 'run_summary.json')

    if os.path.isfile(summary_path):
        with open(summary_path) as f:
            summary = json.load(f)
        print('Best epoch:', summary.get('best_epoch'))
        print('Best mIoU:', summary.get('best_val_miou'))


===== OccCondg_s0 =====
best_val.pth OK
run_summary.json OK
history.jsonl OK
run_config.json OK
Best epoch: 90
Best mIoU: 83.43894958496094

===== OccCondg_s1 =====
best_val.pth OK
run_summary.json OK
history.jsonl OK
run_config.json OK
Best epoch: 87
Best mIoU: 83.59940338134766

===== OccCondg_s2 =====
best_val.pth OK
run_summary.json OK
history.jsonl OK
run_config.json OK
Best epoch: 96
Best mIoU: 83.64779663085938

===== OccCondg_s3 =====
best_val.pth OK
run_summary.json OK
history.jsonl OK
run_config.json OK
Best epoch: 90
Best mIoU: 83.64861297607422

===== OccCondg_s4 =====
best_val.pth OK
run_summary.json OK
history.jsonl OK
run_config.json OK
Best epoch: 98
Best mIoU: 83.51024627685547


In [26]:
import os

ROOT = '/kaggle/working/WeedsGalore_Ipr28/results'

for current, dirs, files in os.walk(ROOT):
    level = current.replace(ROOT, '').count(os.sep)
    indent = '    ' * level

    print(f'{indent}{os.path.basename(current)}/')

    for filename in sorted(files):
        print(f'{indent}    {filename}')

results/
    instance_bins.json
    exp07/
        OccCondg_s0_val_per_image.csv
        OccCondg_s0_val_summary.json
        OccCondg_s1_val_per_image.csv
        OccCondg_s1_val_summary.json
        OccCondg_s2_val_per_image.csv
        OccCondg_s2_val_summary.json
        OccCondg_s3_val_per_image.csv
        OccCondg_s3_val_summary.json
        OccCondg_s4_val_per_image.csv
        OccCondg_s4_val_summary.json
        audit_MCSDg_s0_it1_marginal_by_image.csv
        audit_MCSDg_s0_it1_marginal_summary.json
        audit_MCSDg_s0_it1_predweed_by_image.csv
        audit_MCSDg_s0_it20_marginal_by_image.csv
        audit_MCSDg_s0_it20_marginal_summary.json
        audit_MCSDg_s0_it20_predweed_by_image.csv
        audit_MCSDg_s0_it5_marginal_by_image.csv
        audit_MCSDg_s0_it5_marginal_summary.json
        audit_MCSDg_s0_it5_predweed_by_image.csv
        audit_MCSDg_s1_it1_marginal_by_image.csv
        audit_MCSDg_s1_it1_marginal_summary.json
        audit_MCSDg_s1_it1_predweed_by_i

In [27]:
#Cell K7.5 — Đóng gói kết quả
import csv, shutil, tempfile, zipfile

for run_dir in sorted(glob.glob('runs/OccCondg_s*')):
	if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
		continue
	dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
	os.makedirs(dst, exist_ok=True)
	for filename in ('history.jsonl', 'run_config.json', 'run_summary.json'):
		if os.path.exists(f'{run_dir}/{filename}'):
			shutil.copy(f'{run_dir}/{filename}', dst)

expected_runs = [f'OccCondg_s{seed}' for seed in range(5)]
collector_metrics = (
	'mIoU', 'IoU_bg', 'IoU_crop', 'IoU_weed', 'BIoU_crop', 'BIoU_weed',
	'SmallWeedRecall', 'recall30_small', 'cov_small', 'recall50_medium',
	'recall50_large', 'recall50_all', 'n_small', 'n_medium', 'n_large'
)
for run in expected_runs:
	assert os.path.isfile(f'{OUT}/runs_meta/{run}/run_summary.json'), f'{run}: missing run metadata'
	assert os.path.isfile(f'{OUT}/{run}_val_summary.json'), f'{run}: missing evaluation summary'
	with open(f'{OUT}/{run}_val_summary.json') as f:
		summary = json.load(f)
	assert 'metrics' in summary, f'{run}: evaluation summary has no metrics'
	missing_metrics = [key for key in collector_metrics if key not in summary['metrics']]
	assert not missing_metrics, f'{run}: missing collector metric keys {missing_metrics}'

# The collector expects the full evaluate_ccmc metric schema. Stage only these five
# validation summaries so reference/audit metadata cannot be mistaken for evaluations.
collect_log = f'{OUT}/logs/collect.log'
try:
	with tempfile.TemporaryDirectory(prefix='exp07_collect_') as collect_dir:
		for run in expected_runs:
			shutil.copy2(f'{OUT}/{run}_val_summary.json', collect_dir)
		run_checked(
			[sys.executable, 'src/scripts/collect_results.py',
			 f'--results_dir={collect_dir}', f'--runs_dir={OUT}/runs_meta',
			 f'--out={OUT}/metrics.csv'],
			collect_log
		)
except subprocess.CalledProcessError:
	with open(collect_log) as f:
		print(f.read())
	raise

assert os.path.isfile(f'{OUT}/metrics.csv'), 'Collector did not create metrics.csv'
with open(f'{OUT}/metrics.csv', newline='') as f:
	metric_rows = list(csv.DictReader(f))
assert len(metric_rows) == 5, f'Expected 5 OccCondg evaluation rows, got {len(metric_rows)}'
new_ckpts = ' '.join(sorted(glob.glob('runs/OccCondg_s*/best_val.pth')))
assert len(new_ckpts.split()) == 5
with zipfile.ZipFile('/kaggle/working/exp07_results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
	for root, _, files in os.walk(OUT):
		for filename in files:
			path = os.path.join(root, filename)
			archive.write(path, arcname=path)
with zipfile.ZipFile('/kaggle/working/exp07_occcond_ckpts.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
	for path in new_ckpts.split():
		archive.write(path, arcname=path)
assert os.path.isfile('/kaggle/working/exp07_results.zip')
assert os.path.isfile('/kaggle/working/exp07_occcond_ckpts.zip')
print('/kaggle/working/exp07_results.zip')
print('/kaggle/working/exp07_occcond_ckpts.zip')

/kaggle/working/exp07_results.zip
/kaggle/working/exp07_occcond_ckpts.zip


In [ ]:
#Cell E6.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp06'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
SEG = "--ccmc=False --segformer=True"
TRAIN_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'MCSDg':    f"{SEG} --use_low_level_refine=False --head=mcsd --spectral_guide=True",
    'MCSD':     f"{SEG} --use_low_level_refine=False --head=mcsd",
    'PS':       f"{SEG} --use_low_level_refine=False --head=pixelshuffle",
    'Dnone':    f"{SEG} --use_low_level_refine=True --llr_source=none",
    'LiteSent': f"{SEG} --use_low_level_refine=True --llr_source=raw",
}
EVAL_ARCH = {k: v.replace('--segformer=True ', '') for k, v in TRAIN_ARCH.items()}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E6.1 — Train seed 3, 4 (ưu tiên cặp xác nhận L0/MCSD-g)
ORDER = [('L0', 3), ('MCSDg', 3), ('L0', 4), ('MCSDg', 4), ('MCSD', 3), ('MCSD', 4),
         ('Dnone', 3), ('Dnone', 4), ('LiteSent', 3), ('LiteSent', 4), ('PS', 3), ('PS', 4)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E6.2 — Đánh giá validation, seed 0–4
def evaluate(model, seed, arch, extra='', tag=None):
    ck = f'runs/{model}_s{seed}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', model, seed); return
    name = f'{tag or model}_s{seed}'
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={name} --out_prefix={OUT}/{name}_val > {OUT}/logs/eval_{name}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{name}.log

for model in ('L0', 'MCSDg', 'MCSD', 'PS', 'Dnone', 'LiteSent'):
    for s in range(5):
        evaluate(model, s, EVAL_ARCH[model])
# Ablation: số vòng Sinkhorn lúc suy luận (cùng checkpoint MCSD-g)
for it in (1, 20):
    for s in range(5):
        evaluate('MCSDg', s, EVAL_ARCH['MCSDg'], f'--sinkhorn_iters={it}', tag=f'MCSDgIt{it}')

In [ ]:
#Cell E6.3 — Xác nhận (seed 3–4) và bảng 5 seed
def compare(a, b, gate='none', seeds=range(5), extra='', name=None):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    name = name or f'{a}_vs_{b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} {extra} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

# 1) XÁC NHẬN trên seed mới
compare('MCSDg', 'L0', 'abs', seeds=(3, 4), extra='--min_delta=0 --min_seeds_pass=2', name='CONFIRM_MCSDg_vs_L0_s34')
# 2) Bảng 5 seed
compare('MCSDg', 'L0', 'abs')
compare('MCSD', 'L0', 'abs')
compare('MCSD', 'PS', 'g4')
compare('MCSDg', 'MCSD')
compare('PS', 'L0')
compare('Dnone', 'L0')
compare('LiteSent', 'L0')
compare('MCSDg', 'Dnone')
compare('MCSDg', 'LiteSent')
# 3) Ablation số vòng Sinkhorn
compare('MCSDgIt1', 'MCSDg')
compare('MCSDgIt20', 'MCSDg')

In [ ]:
#Cell E6.4 — Bảo toàn khối lượng theo số vòng Sinkhorn (MCSD-g seed 0)
for it in (1, 5, 20):
    !python src/scripts/mass_check.py --dataset_path={DS} --ckpt=runs/MCSDg_s0/best_val.pth --ccmc=False --head=mcsd --spectral_guide=True --sinkhorn_iters={it} --out={OUT}/mass_check_MCSDg_s0_it{it}.json

In [ ]:
#Cell E6.5 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E5.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp05'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
SEG = "--ccmc=False --segformer=True --use_low_level_refine=False"
TRAIN_ARCH = {
    'MCSD':  f"{SEG} --head=mcsd",
    'PS':    f"{SEG} --head=pixelshuffle",
    'MCSDg': f"{SEG} --head=mcsd --spectral_guide=True",
}
EVAL_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'Dnone':    "--ccmc=False --use_low_level_refine=True --llr_source=none",
    'LiteSent': "--ccmc=False --use_low_level_refine=True --llr_source=raw",
    'MCSD':     "--ccmc=False --use_low_level_refine=False --head=mcsd",
    'PS':       "--ccmc=False --use_low_level_refine=False --head=pixelshuffle",
    'MCSDg':    "--ccmc=False --use_low_level_refine=False --head=mcsd --spectral_guide=True",
}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E5.1 — Train (theo thứ tự ưu tiên)
ORDER = [('MCSD', 0), ('PS', 0), ('MCSD', 1), ('PS', 1), ('MCSD', 2), ('PS', 2),
         ('MCSDg', 0), ('MCSDg', 1), ('MCSDg', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E5.2 — Đánh giá validation (mô hình mới + tham chiếu)
def evaluate(run, arch):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} --bins_json={BINS} --boundary_px=3 --run_name={run} --out_prefix={OUT}/{run}_val > {OUT}/logs/eval_{run}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{run}.log

for model in ('L0', 'Dnone', 'LiteSent', 'MCSD', 'PS', 'MCSDg'):
    for s in (0, 1, 2):
        evaluate(f'{model}_s{s}', EVAL_ARCH[model])

In [ ]:
#Cell E5.3 — Gate và so sánh
def compare(a, b, gate='none', seeds=(0, 1, 2)):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{a}_vs_{b}.json 2>&1 | tee {OUT}/logs/cmp_{a}_vs_{b}.log

compare('MCSD', 'L0', 'abs')       # H-MCSD
compare('MCSD', 'PS', 'g4')        # H-mech
compare('PS', 'L0')                # head sub-pixel không bảo toàn khối lượng
compare('MCSD', 'Dnone')           # H-eff
compare('MCSD', 'LiteSent')
compare('MCSDg', 'MCSD')           # H-spec
compare('MCSDg', 'L0', 'abs')

In [ ]:
#Cell E5.4 — Profile (600×600, cùng quy ước)
for label in ('MCSD', 'PS', 'MCSDg'):
    !python src/scripts/profile_model.py {EVAL_ARCH[label]} --label={label} --out={OUT}/profile_{label}.json 2>&1 | grep -E "GFLOPs|latency_ms_median|params_M"

In [ ]:
%%writefile /kaggle/working/WeedsGalore_Ipr28/mass_check.py
import json, sys
import numpy as np, torch
sys.path.insert(0, 'src')
from nets.mcsd import MCSDSegFormer
from ccmc_common import load_checkpoint
from datasets import WeedsGaloreDataset
ds_path, ckpt, out = sys.argv[1], sys.argv[2], sys.argv[3]
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = load_checkpoint(MCSDSegFormer(head='mcsd'), ckpt, dev).to(dev).eval()
ds = WeedsGaloreDataset(ds_path, None, 5, 3, False, 'val', False)
errs, pred_mass, gt_mass = [], 0.0, 0.0
with torch.no_grad():
    for i in range(len(ds)):
        img, _, lab = ds[i]
        x = torch.from_numpy(np.ascontiguousarray(img)).float()[None].to(dev)
        logp, ex = net(x, return_aux=True)
        p = logp.exp()
        q = torch.softmax(ex['occ_logits'].float(), 1)
        errs.append((torch.nn.functional.avg_pool2d(p, 4) - q).abs().max().item())
        pred_mass += p[:, 2].sum().item()
        gt_mass += float((np.asarray(lab) == 2).sum())
res = {'max_row_marginal_error': max(errs), 'mean_row_marginal_error': float(np.mean(errs)),
       'pred_weed_mass_over_gt': pred_mass / gt_mass}
print(res)
json.dump(res, open(out, 'w'), indent=2)

In [ ]:
!python mass_check.py {DS} runs/MCSD_s0/best_val.pth {OUT}/mass_check_MCSD_s0.json

In [ ]:
#Cell E5.6 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E4.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp04'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
TRAIN_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'LiteSent': "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=raw",
    'Dnone':    "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=none",
    'DF1':      "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=f1",
}
EVAL_ARCH = {k: v.replace('--segformer=True ', '') for k, v in TRAIN_ARCH.items()}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E4.1 — Train
ORDER = [('Dnone', 0), ('DF1', 0), ('LiteSent', 1), ('Dnone', 1), ('DF1', 1),
         ('LiteSent', 2), ('Dnone', 2), ('DF1', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/best_val.pth') and (os.path.islink(f'runs/{run}/best_val.pth')
                                                       or os.path.exists(f'runs/{run}/run_summary.json')):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E4.2 — Đánh giá validation (cả L0 0–2 và LiteSent s0 lấy từ các Exp trước)
def evaluate(run, arch):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} --bins_json={BINS} --boundary_px=3 --run_name={run} --out_prefix={OUT}/{run}_val > {OUT}/logs/eval_{run}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{run}.log

for model in ('L0', 'LiteSent', 'Dnone', 'DF1'):
    for s in (0, 1, 2):
        evaluate(f'{model}_s{s}', EVAL_ARCH[model])

In [ ]:
#Cell E4.3 — So sánh (bootstrap phân tầng, 3 seed)
def compare(a, b, gate='none', seeds=(0, 1, 2)):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{a}_vs_{b}.json 2>&1 | tee {OUT}/logs/cmp_{a}_vs_{b}.log

compare('LiteSent', 'L0')      # tổng lợi ích của LLR (3 seed)
compare('Dnone', 'L0')         # chỉ tăng độ phân giải decoder
compare('LiteSent', 'Dnone')   # giá trị thêm của chi tiết từ băng thô
compare('LiteSent', 'DF1')     # băng thô so với F1
compare('DF1', 'Dnone')        # F1 có thêm gì không

In [ ]:
#Cell E4.4 — Profile (600×600, cùng quy ước)
for label in ('Dnone', 'DF1', 'LiteSent'):
    !python src/scripts/profile_model.py {EVAL_ARCH[label]} --label={label} --out={OUT}/profile_{label}.json 2>&1 | grep -E "GFLOPs|latency_ms_median|params_M"

In [ ]:
#Cell E4.5 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E3.0 — Biến dùng chung, nối checkpoint cũ
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp03'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
BASE = "--ccmc=True --use_low_level_refine=False"
LAM_LEARN = "--lam_init=0.5 --lam_lr=0.01 --learn_lambda=True"
LAM_FIXED1 = "--lam_init=0.999 --learn_lambda=False"   # sigmoid(logit(0.999)) ≈ 1
ARCH = {
    'L0':    f"{BASE} --prior_mode=none",
    'C3a':   f"{BASE} --prior_mode=gt_weed --injection=ccmc_sr {LAM_LEARN}",
    'C2a':   f"{BASE} --prior_mode=gt_veg  --injection=ccmc_sr {LAM_LEARN}",
    'C3f':   f"{BASE} --prior_mode=gt_weed --injection=ccmc_sr {LAM_FIXED1}",
    'C3K5a': f"{BASE} --prior_mode=gt_weed --injection=pre_sr_gate {LAM_LEARN}",
}
# Cờ chỉ dùng cho train (evaluate_ccmc không có lam_*)
EVAL_ARCH = {k: ' '.join(t for t in v.split() if not t.startswith(('--lam_', '--learn_lambda')))
             for k, v in ARCH.items()}
# Nối checkpoint Exp01/Exp02 (chỉ đọc)
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))

In [ ]:
#Cell E3.1 — Bin (phải trùng [45, 147])
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
# Cell E3.2 — Train (theo thứ tự ưu tiên)
ORDER = [('L0', 2), ('C3a', 0), ('C2a', 0), ('C3f', 0), ('C3a', 1), ('C2a', 1), ('C3a', 2), ('C2a', 2),
         ('C3K5a', 0), ('C3K5a', 1), ('C3K5a', 2), ('C3f', 1), ('C3f', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2
    !grep -E "lambdas" {OUT}/logs/train_{run}.log | tail -n 1

In [ ]:
#Cell E3.3 — Quỹ đạo λ (kiểm tra λ đã thực sự di chuyển)
for h in sorted(glob.glob('runs/*/history.jsonl')):
    rows = [json.loads(l) for l in open(h)]
    if not rows[0]['lambdas']:
        continue
    lam = [r['lambdas'] for r in rows]; best = rows[-1]['best_epoch']
    print(f"{h.split('/')[1]:10s} ep0={[round(v,3) for v in lam[0]]} ep10={[round(v,3) for v in lam[min(10,len(lam)-1)]]} "
          f"ep50={[round(v,3) for v in lam[min(50,len(lam)-1)]]} best(ep{best})={[round(v,3) for v in lam[best]]}")

In [ ]:
#Cell E3.4 — Đánh giá validation
def evaluate(run, arch, extra='', tag=None):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    tag = tag or run
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={tag} --out_prefix={OUT}/{tag}_val > {OUT}/logs/eval_{tag}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{tag}.log

for s in (0, 1, 2):
    evaluate(f'L0_s{s}', EVAL_ARCH['L0'])
for model, seed in ORDER:
    if model != 'L0':
        evaluate(f'{model}_s{seed}', EVAL_ARCH[model])

In [ ]:
#Cell E3.5 — Cơ chế có hoạt động không? (cùng checkpoint, chỉ đổi prior lúc suy luận)
for model in ('C3a', 'C3f'):
    for s in (0, 1, 2):
        run = f'{model}_s{s}'
        evaluate(run, EVAL_ARCH[model], '--prior_override=uniform', tag=f'{run}_ovr-uniform')
for noise in ('dilate:3', 'drop:0.3', 'shift:4'):
    evaluate('C3a_s0', EVAL_ARCH['C3a'], f'--prior_noise={noise}', tag=f"C3a_s0_noise-{noise.replace(':','')}")

In [ ]:
#Cell E3.6 — So sánh (bootstrap phân tầng theo seed + ảnh) và A/A test
def compare(a, b, gate, seeds=(0, 1, 2), suffix_a='', suffix_b='', name=None, extra=''):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}{suffix_a}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}{suffix_b}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}{suffix_a}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}{suffix_b}_val_per_image.csv' for s in common)
    name = name or f'{a}{suffix_a}_vs_{b}{suffix_b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a}{suffix_a} --name_b={b}{suffix_b} --gate={gate} {extra} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

# 1) Cơ chế: prior GT so với prior đồng đều trên CÙNG checkpoint
compare('C3a', 'C3a', 'none', suffix_b='_ovr-uniform', name='C3a_activity')
compare('C3f', 'C3f', 'none', suffix_b='_ovr-uniform', name='C3f_activity')
# 2) G2-screen
compare('C3a', 'C2a', 'screen')
# 3) Trần mô tả và đối chứng cùng leakage
compare('C3a', 'L0', 'none'); compare('C3f', 'L0', 'none'); compare('C2a', 'L0', 'none')
compare('C3a', 'C3K5a', 'g4')
# 4) A/A test: cùng mô hình L0, khác seed, chỉ bootstrap theo ảnh
!python src/scripts/compare_runs.py --a={OUT}/L0_s0_val_per_image.csv --b={OUT}/L0_s1_val_per_image.csv --name_a=L0_s0 --name_b=L0_s1 --seed_bootstrap=False --out={OUT}/cmp_AA_L0s0_vs_L0s1.json 2>&1 | tee {OUT}/logs/cmp_AA.log

In [ ]:
#Cell E3.7 — Gom kết quả và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if not os.path.exists(f'{run_dir}/history.jsonl') or os.path.islink(f'{run_dir}/best_val.pth'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E2.0 — Biến dùng chung và khôi phục checkpoint Exp01
import os, glob, json, zipfile
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp02'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
L0_ARCH = "--ccmc=True --prior_mode=none --use_low_level_refine=False"
ARCH = {
    'C2':   "--ccmc=True --prior_mode=gt_veg  --injection=ccmc_sr      --use_low_level_refine=False",
    'C3':   "--ccmc=True --prior_mode=gt_weed --injection=ccmc_sr      --use_low_level_refine=False",
    'C3K5': "--ccmc=True --prior_mode=gt_weed --injection=pre_sr_gate  --use_low_level_refine=False",
    'C3K6': "--ccmc=True --prior_mode=gt_weed --injection=post_sr_gate --use_low_level_refine=False",
}
# Exp01 checkpoints (zip hoặc thư mục đã giải nén trong /kaggle/input)
for z in glob.glob('/kaggle/input/**/exp01_ckpts.zip', recursive=True):
    zipfile.ZipFile(z).extractall('.')
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print('Exp01 ckpts:', sorted(glob.glob('runs/*/best_val.pth')))

In [ ]:
#Cell E2.1 — Bin kích thước (phải trùng Exp01: edges = [45, 147])
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
edges = json.load(open(BINS))['edges']
print('edges', edges)
assert edges == [45.0, 147.0], 'Bin khác Exp01 -> dừng lại và gửi logs/bins.log'
!cp {BINS} {OUT}/instance_bins.json

In [ ]:
#Cell E2.2 — Train C2, C3, C3-K5, C3-K6 (seed 0 rồi seed 1)
ORDER = [('C3', 0), ('C2', 0), ('C3K5', 0), ('C3', 1), ('C2', 1), ('C3K6', 0), ('C3K5', 1), ('C3K6', 1)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|lambdas|Traceback|Error" | tail -n 3

In [ ]:
#Cell E2.3 — Đánh giá validation (Boundary IoU 3 px)
def evaluate(run, arch, extra=''):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    tag = run + (extra.replace('--prior_noise=', '_noise-').replace(':', '') if extra else '')
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={tag} --out_prefix={OUT}/{tag}_val > {OUT}/logs/eval_{tag}.log 2>&1
    !grep -E "mIoU|small|GT prior" {OUT}/logs/eval_{tag}.log

# L0 / LiteSent đánh giá lại (BIoU 3 px), từ checkpoint Exp01
for seed in (0, 1):
    evaluate(f'L0_s{seed}', L0_ARCH)
evaluate('LiteSent_s0', '--ccmc=False --use_low_level_refine=True')
# Oracle models (GT leakage)
for model, seed in ORDER:
    evaluate(f'{model}_s{seed}', ARCH[model])

In [ ]:
#Cell E2.4 — Độ nhạy với prior GT bị nhiễu (C3 seed 0, cùng checkpoint)
for noise in ('dilate:3', 'erode:1', 'drop:0.3', 'shift:4'):
    evaluate('C3_s0', ARCH['C3'], f'--prior_noise={noise}')

In [ ]:
# Cell E2.4b — λ cuối và quỹ đạo λ của các mô hình oracle
import json, glob
for h in sorted(glob.glob('runs/C*/history.jsonl')):
    rows = [json.loads(l) for l in open(h)]
    lam = [r['lambdas'] for r in rows]
    best = rows[-1]['best_epoch']
    print(f"{h.split('/')[1]:9s} λ@ep0={[round(v,3) for v in lam[0]]} "
          f"λ@ep50={[round(v,3) for v in lam[min(50,len(lam)-1)]]} "
          f"λ@best(ep{best})={[round(v,3) for v in lam[best]]} λ@last={[round(v,3) for v in lam[-1]]}")

In [ ]:
#Cell E2.5 — So sánh paired và gate
def compare(a, b, gate, seeds=(0, 1)):
    # chỉ dùng các seed có ở CẢ HAI mô hình (paired)
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, 'vs', b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    name = f'{a}_vs_{b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

compare('C3', 'C2', 'screen')      # G2-screen (một chiều)
compare('C3', 'C3K5', 'g4')        # cùng leakage: CCMC-SR so với gate trước SR
compare('C3', 'C3K6', 'g4')        # cùng leakage: CCMC-SR so với gate sau SR
compare('C3', 'L0', 'none')        # biên độ trần, chỉ mô tả
compare('C2', 'L0', 'none')

In [ ]:
#Cell E2.6 — Gom kết quả và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if not os.path.exists(f'{run_dir}/run_config.json'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
!zip -qr /kaggle/working/{EXP}_ckpts.zip runs/C*/best_val.pth
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E1.0 — Biến dùng chung
import os
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp01'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'   # bin cố định cho MỌI thí nghiệm sau này
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
L0_ARCH = "--ccmc=True --prior_mode=none --use_low_level_refine=False"
LITE_ARCH = "--ccmc=False --segformer=True --use_low_level_refine=True"
print(COMMON)

In [ ]:
#Cell E1.1 — Bin kích thước cỏ (tam phân vị của train, khóa trước khi xem mô hình)
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} 2>&1 | tee {OUT}/logs/bins.log | head -n 60
!cp {BINS} {OUT}/instance_bins.json

In [ ]:
#Cell E1.2 — Train L0 seed 0 và seed 1
for seed in (0, 1):
    run = f'L0_s{seed}'
    !python src/train.py {COMMON} {L0_ARCH} --seed={seed} --run_name=L0__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.3 — Train LiteSent seed 0 (cùng pipeline, để làm Diag-D0 trên checkpoint có LLR)
!python src/train.py {COMMON} {LITE_ARCH} --seed=0 --run_name=LiteSent__s0 --out_dir=runs/LiteSent_s0 2>&1 | tee {OUT}/logs/train_LiteSent_s0.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.4 — Diag-D0: đánh giá validation (IoU, Boundary IoU, recall theo bin, theo ngày)
for seed in (0, 1):
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt=runs/L0_s{seed}/best_val.pth {L0_ARCH} --bins_json={BINS} --run_name=L0__s{seed} --out_prefix={OUT}/L0_s{seed}_val 2>&1 | tee {OUT}/logs/eval_L0_s{seed}.log
if os.path.exists('runs/LiteSent_s0/best_val.pth'):
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt=runs/LiteSent_s0/best_val.pth --ccmc=False --use_low_level_refine=True --bins_json={BINS} --run_name=LiteSent__s0 --out_prefix={OUT}/LiteSent_s0_val 2>&1 | tee {OUT}/logs/eval_LiteSent_s0.log

In [ ]:
#Cell E1.5 — Diag-D0: recall cỏ nhỏ so với cỏ lớn trong cùng mô hình (bootstrap theo ảnh)
import csv, json, numpy as np
def small_vs_large(path, n_boot=10000, seed=0):
    rows = list(csv.DictReader(open(path)))
    g = lambda k: np.array([float(r[k]) for r in rows])
    out = {}
    for t in ('50', '30'):
        tp_s, n_s, tp_l, n_l = g(f'tp{t}_small'), g('n_small'), g(f'tp{t}_large'), g('n_large')
        point = 100 * (tp_l.sum() / n_l.sum() - tp_s.sum() / n_s.sum())
        rng = np.random.default_rng(seed)
        W = rng.multinomial(len(rows), np.full(len(rows), 1 / len(rows)), size=n_boot)
        with np.errstate(invalid='ignore', divide='ignore'):
            boot = 100 * ((W @ tp_l) / (W @ n_l) - (W @ tp_s) / (W @ n_s))
        boot = boot[np.isfinite(boot)]
        out[f'recall{t}_large_minus_small'] = {'delta': point, 'ci95': np.percentile(boot, [2.5, 97.5]).tolist()}
    return out
d0 = {}
for name in ['L0_s0', 'L0_s1', 'LiteSent_s0']:
    p = f'{OUT}/{name}_val_per_image.csv'
    if os.path.exists(p):
        d0[name] = small_vs_large(p)
print(json.dumps(d0, indent=2))
json.dump(d0, open(f'{OUT}/d0_small_vs_large.json', 'w'), indent=2)

In [ ]:
#Cell E1.6 — LiteSent so với L0 (paired bootstrap, seed 0, mang tính bổ trợ)
if os.path.exists(f'{OUT}/LiteSent_s0_val_per_image.csv'):
    !python src/scripts/compare_runs.py --a={OUT}/LiteSent_s0_val_per_image.csv --b={OUT}/L0_s0_val_per_image.csv --name_a=LiteSent --name_b=L0 --out={OUT}/cmp_LiteSent_vs_L0_s0.json 2>&1 | tee {OUT}/logs/cmp_LiteSent_vs_L0.log

In [ ]:
#Cell E1.7 — Diag-D2: linear probe trước/sau SR (stage 1–3)
for seed in (0, 1):
    !python src/scripts/probe_sr.py --dataset_path={DS} --ckpt=runs/L0_s{seed}/best_val.pth {L0_ARCH} --bins_json={BINS} --out_prefix={OUT}/d2_probe_L0_s{seed} 2>&1 | tee {OUT}/logs/probe_L0_s{seed}.log | tail -n 60

In [ ]:
#Cell E1.8 — Profile (cùng quy ước: 1×5×600×600, fp32, batch 1, T4)
profiles = {
    'L0': L0_ARCH,
    'LiteSent': LITE_ARCH.replace('--segformer=True ', ''),
    'L3': '--ccmc=True --prior_mode=learned_weed --injection=ccmc_sr --use_low_level_refine=False',
    'K5': '--ccmc=True --prior_mode=learned_weed --injection=pre_sr_gate --use_low_level_refine=False',
    'K6': '--ccmc=True --prior_mode=learned_weed --injection=post_sr_gate --use_low_level_refine=False',
}
for label, arch in profiles.items():
    !python src/scripts/profile_model.py {arch} --label={label} --out={OUT}/profile_{label}.json 2>&1 | tail -n 3

In [ ]:
#Cell E1.9 — Gom kết quả và đóng gói để gửi review
import shutil, glob
for run_dir in sorted(glob.glob('runs/*')):
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
shutil.copytree('results/env', f'{OUT}/env', dirs_exist_ok=True)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv
!cd /kaggle/working/WeedsGalore_Ipr28 && zip -qr /kaggle/working/{EXP}_results.zip {OUT} {BINS}
!cd /kaggle/working/WeedsGalore_Ipr28 && zip -qr /kaggle/working/{EXP}_ckpts.zip runs/*/best_val.pth
!ls -lh /kaggle/working/*.zip

In [ ]:
!pip install thop

In [ ]:
#Cell chạy E1/E2:
!python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --dataset_size_train=104 \
    --segformer=True \
    --spectral_guided=False \
    --use_spectral_indices=False \
    --use_attention=False \
    --dlv3p_do=False \
    --in_channels=5 \
    --num_classes=3 \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --batch_size=8 \
    --num_workers=4 \
    --lr=0.0003 \
    --weight_bg=0.9 \
    --weight_crop=1.0 \
    --weight_weed=1.3 \
    --use_dice_loss=True \
    --dice_weight=0.3 \
    --epochs=100 \
    --seed=0 \
    --out_dir=/kaggle/working/out_dir_segformer_no_refine_seed0 \
    --log_interval=25 \
    --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_segformer_no_refine_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=5 \
    --num_classes=3 \
    --temperature=1.0

In [ ]:
#Cell E5, RGB-only nhiều seed:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=3 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_rgb_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=5 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_5band_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
from pathlib import Path

file = Path("/kaggle/working/WeedsGalore_Ipr28/src/evaluate.py")
text = file.read_text()

old = 'print("Architecture: 5-band SegFormer")'
new = '''print(f"Architecture: SegFormer ({FLAGS.in_channels}-channel, "
              f"LowLevelRefine={FLAGS.use_low_level_refine})")'''

assert old in text, "Không tìm thấy dòng cũ hoặc file đã thay đổi!"

file.write_text(text.replace(old, new, 1))
print("Đã cập nhật thành công dòng 65!")

In [ ]:
%cd /kaggle/working/WeedsGalore_Ipr28

import subprocess
import sys

cmd = [
    sys.executable,
    "src/calibrate_temperature.py",
    "--dataset_path", "/kaggle/working/weedsgalore-dataset",
    "--ckpt", "/kaggle/working/out_dir_rgb_seed0/best_val.pth",
    "--segformer=True",
    "--pretrained_backbone=True",
    "--spectral_guided=False",
    "--in_channels", "3",
    "--num_classes", "3",
    "--use_low_level_refine=True",
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("Exit code:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout)
print("\n===== STDERR =====")
print(result.stderr)

In [ ]:
import subprocess
import sys
import re
import numpy as np
from pathlib import Path

REPO = "/kaggle/working/WeedsGalore_Ipr28"
DATASET = "/kaggle/working/weedsgalore-dataset"

# Trong mọi subprocess, chạy từ thư mục repository
def run_command(command):
    result = subprocess.run(
        command,
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout + result.stderr

experiments = {
    "RGB": [
        f"/kaggle/working/out_dir_rgb_seed{i}/best_val.pth"
        for i in range(5)
    ],
    "5-band": [
        f"/kaggle/working/out_dir_5band_seed{i}/best_val.pth"
        for i in range(5)
    ],
}

results = {}

for name, checkpoints in experiments.items():
    miou_values = []
    ece_values = []
    temperatures = []

    in_channels = 3 if name == "RGB" else 5

    for seed, checkpoint in enumerate(checkpoints):
        if not Path(checkpoint).exists():
            print(f"SKIP: {checkpoint}")
            continue

        common = [
            f"--dataset_path={DATASET}",
            f"--ckpt={checkpoint}",
            "--segformer=True",
            "--pretrained_backbone=True",
            "--spectral_guided=False",
            f"--in_channels={in_channels}",
            "--num_classes=3",
            "--use_low_level_refine=True",
        ]

        calibration_log = run_command(
            [sys.executable, "src/calibrate_temperature.py"] + common
        )

        match = re.search(
            r"Fitted temperature:\s*T\s*=\s*([0-9.]+)",
            calibration_log,
        )
        if match is None:
            raise RuntimeError(
                f"Không đọc được temperature cho {name}, seed {seed}"
            )

        temperature = float(match.group(1))
        temperatures.append(temperature)

        evaluation_log = run_command([
            sys.executable,
            "src/evaluate.py",
            "--split=test",
            *common,
            f"--temperature={temperature}",
        ])

        print(evaluation_log)

        miou_match = re.search(r"mIoU:\s*([0-9.]+)%", evaluation_log)
        ece_match = re.search(r"ECE:\s*(?:tensor\()?([0-9.]+)", evaluation_log)

        if miou_match is None or ece_match is None:
            raise RuntimeError(f"Không đọc được kết quả cho {name}, seed {seed}")

        miou_values.append(float(miou_match.group(1)))
        ece_values.append(float(ece_match.group(1)))

    results[name] = {
        "mIoU": miou_values,
        "ECE": ece_values,
        "T": temperatures,
    }

print("\n===== SUMMARY =====")
for name, values in results.items():
    print(f"\n{name}")
    print("Temperatures:", values["T"])
    print(
        f"Test mIoU: {np.mean(values['mIoU']):.2f} "
        f"+/- {np.std(values['mIoU'], ddof=1):.2f}%"
    )
    print(
        f"Test ECE: {np.mean(values['ECE']):.5f} "
        f"+/- {np.std(values['ECE'], ddof=1):.5f}"
    )